In [ ]:
# ============================================================
# 라이브러리 설치 및 import (캐글 환경)
# ============================================================
# 본 노트북에서 사용하는 주요 라이브러리는 다음과 같습니다.
#   lightgbm           : 그래디언트 부스팅 결정 트리 모델. 표 형식 데이터에서 좋은 성능을 보입니다.
#   catboost           : 범주형 변수를 내부적으로 자연스럽게 다루는 부스팅 모델.
#   shap               : 모델 해석을 위한 SHAP 값 계산 라이브러리.
#   fonts-nanum        : 캐글 환경에서 한글 그래프를 위한 NanumGothic 폰트.
# 
# 캐글 환경에서는 lightgbm, catboost가 기본 설치되어 있으므로 빠르게 통과합니다.
# 한글 폰트는 캐글 기본 환경에 없을 수 있어 명시적으로 설치합니다.
!pip install shap --quiet
!apt-get install -y fonts-nanum > /dev/null 2>&1

import warnings
warnings.filterwarnings('ignore')   # 학습 과정에서 출력되는 경고 메시지를 숨깁니다.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.font_manager as fm
import seaborn as sns
import lightgbm as lgb

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold       # 클래스 비율을 유지하며 fold를 나누는 도구
from sklearn.metrics import roc_auc_score                 # 평가 지표
from sklearn.preprocessing import MultiLabelBinarizer     # 콤마로 구분된 멀티라벨을 0/1 컬럼으로 분해

# ------------------------------------------------------------
# 캐글 환경에서 한글 폰트 적용
# ------------------------------------------------------------
# fonts-nanum 패키지로 설치된 NanumGothic 폰트를 matplotlib에 등록합니다.
# 폰트 캐시 갱신을 위해 fontManager에 폰트 파일을 직접 추가합니다.
nanum_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
import os
if os.path.exists(nanum_path):
    fm.fontManager.addfont(nanum_path)
    plt.rcParams['font.family'] = 'NanumGothic'
    print('한글 폰트(NanumGothic) 적용 완료')
else:
    plt.rcParams['font.family'] = 'DejaVu Sans'
    print('NanumGothic 폰트 설치 실패. 영문 폰트로 진행합니다.')

plt.rcParams['axes.unicode_minus'] = False               # 그래프에서 마이너스 부호가 깨지지 않도록 설정

# 재현성 확보를 위해 시드를 고정합니다.
RANDOM_STATE = 42
TARGET = '임신 성공 여부'
np.random.seed(RANDOM_STATE)

print('환경 설정이 완료되었습니다.')

In [ ]:
# ============================================================
# 캐글 환경 — Drive 마운트 불필요
# ============================================================
# 코랩에서는 Google Drive를 마운트해야 했지만,
# 캐글에서는 데이터셋이 자동으로 /kaggle/input/ 아래에 마운트됩니다.
# 별도의 마운트 작업이 필요 없으므로 안내 문구만 출력합니다.

print('캐글 환경 — Drive 마운트 불필요')
print('데이터는 /kaggle/input/ 아래에서 자동으로 접근 가능합니다.')

In [ ]:
# ============================================================
# 데이터 로드 및 실험 옵션 토글 (캐글 환경)
# ============================================================
# 본 노트북의 모든 실험 옵션은 아래 세 변수로 통제됩니다.

# ------------------------------------------------------------
# 캐글 데이터셋 경로
# ------------------------------------------------------------
# 캐글에서는 사용자가 추가한 데이터셋이 /kaggle/input/<dataset-slug>/ 형태로 마운트됩니다.
# 본인의 데이터셋 슬러그(slug)를 정확히 확인하시고, 필요시 아래 경로를 조정하세요.
# 우측 Input 패널에서 폴더 위에 마우스를 올리면 정확한 경로가 표시됩니다.
DATA_DIR = '//kaggle/input/datasets/min2602/my-data/'  

# 캐글 작업 폴더 (제출 파일과 시각화 PNG가 저장될 곳)
OUTPUT_DIR = '/kaggle/working/'

# ------------------------------------------------------------
# 실험 옵션 토글 (이 세 가지가 본 노트북의 핵심 설정입니다)
# ------------------------------------------------------------
# N_SPLITS         : K-Fold의 K값. 본 노트북은 25-fold로 설정합니다.
#                    25이면 학습 폴드가 96%로 늘어 데이터 활용이 늘지만 학습 시간이 길어집니다.
# USE_RANK_AVG     : 앙상블 시 각 모델 예측을 순위(rank)로 변환한 후 평균할지 결정합니다.
#                    AUC는 순위 기반 지표이므로, 모델 간 확률 분포가 다를 때 도움이 됩니다.
# USE_DAY45_PARAMS : day4_5 실험에서 탐색한 100회 Optuna 결과 파라미터로 교체할지 결정합니다.
#                    참고용 비교 옵션이며, 기본값(v4 파라미터)이 가장 안정적입니다.

N_SPLITS         = 25
USE_RANK_AVG     = True
USE_DAY45_PARAMS = False

# ------------------------------------------------------------
# GPU 사용 가능 여부 자동 감지
# ------------------------------------------------------------
# 캐글에서 우측 Settings에 'Accelerator: GPU T4 x2'를 선택했다면 GPU가 활성화됩니다.
# CatBoost는 GPU에서 약 3~5배 빠르게 학습됩니다.
import subprocess
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=5)
    USE_GPU = result.returncode == 0
except Exception:
    USE_GPU = False

if USE_GPU:
    print('[GPU 감지] CatBoost를 GPU로 가속합니다.')
else:
    print('[CPU 모드] GPU가 감지되지 않아 CPU로 학습합니다.')

# ------------------------------------------------------------
# 데이터 파일 로드
# ------------------------------------------------------------
df_raw      = pd.read_csv(DATA_DIR + 'train.csv')
df_test_raw = pd.read_csv(DATA_DIR + 'test.csv')

# 컬럼명 양 끝에 보이지 않는 공백이 끼어 있는 경우가 있어 일괄 정리합니다.
# 이 처리를 빠뜨리면 같은 이름처럼 보이는 컬럼이 다르게 인식되어 디버깅이 매우 까다로워집니다.
df_raw.columns      = [c.strip() for c in df_raw.columns]
df_test_raw.columns = [c.strip() for c in df_test_raw.columns]

print(f'학습 데이터: {df_raw.shape[0]:,}행 × {df_raw.shape[1]}열')
print(f'평가 데이터: {df_test_raw.shape[0]:,}행 × {df_test_raw.shape[1]}열')
print()
print('현재 실험 설정')
print(f'  N_SPLITS         = {N_SPLITS}')
print(f'  USE_RANK_AVG     = {USE_RANK_AVG}')
print(f'  USE_DAY45_PARAMS = {USE_DAY45_PARAMS}')
print(f'  USE_GPU          = {USE_GPU}')
print()
print('타겟 분포 (임신 성공 여부)')
print(df_raw[TARGET].value_counts())
print()
print('실패와 성공의 비율이 약 2.87 대 1로, 클래스 불균형이 존재합니다.')
print('이 부분은 모델 학습 시 scale_pos_weight로 보정합니다.')

In [ ]:
# ============================================================
# EDA 전용 DataFrame 준비
# ============================================================
# 시각화에서 필요한 변환을 모델 학습용 데이터와 분리하기 위해
# 별도의 eda_df를 만들어 그곳에서만 작업합니다.

eda_df = df_raw.copy()
eda_df.columns = [c.strip() for c in eda_df.columns]

# ------------------------------------------------------------
# 나이 변수 수치화 (시각화용)
# ------------------------------------------------------------
# 원본은 '만35-37세'와 같은 구간 형태의 문자열입니다.
# 시각화에서 비교하기 쉽도록 구간의 중앙값과 임상 위험도 분류를 함께 부여합니다.
# 임상적으로 만 35세 이상부터 자연 임신 성공률이 의미 있게 떨어지므로 '고위험'으로,
# 만 40세 이상은 '초고위험'으로 분류했습니다. 만 45세 이상은 자가 난자 임신이 거의 어렵고
# 기증 난자 비율이 높아지는 구간이라 별도로 보아야 합니다.

age_info_eda = {
    '만18-34세': {'val': 26.0, 'risk': '정상_임신군'},
    '만35-37세': {'val': 36.0, 'risk': '고위험_임신군'},
    '만38-39세': {'val': 38.5, 'risk': '고위험_임신군'},
    '만40-42세': {'val': 41.0, 'risk': '초고위험_임신군'},
    '만43-44세': {'val': 43.5, 'risk': '초고위험_임신군'},
    '만45-50세': {'val': 47.5, 'risk': '초고위험_임신군'},
    '알 수 없음': {'val': np.nan, 'risk': '미분류'},
}
eda_df['나이_수치'] = eda_df['시술 당시 나이'].map(
    lambda x: age_info_eda.get(x, {'val': np.nan})['val'])
eda_df['임신_위험도_범주'] = eda_df['시술 당시 나이'].map(
    lambda x: age_info_eda.get(x, {'risk': '미분류'})['risk'])

# ------------------------------------------------------------
# 시술 유형 단순 분류 (시각화용)
# ------------------------------------------------------------
# 원본의 '특정 시술 유형'은 항목이 매우 다양해 시각화에 그대로 쓰기 어렵습니다.
# 임상적으로 의미 있는 네 가지 큰 그룹으로 묶어 사용합니다.
#   Blastocyst_Transfer : 배반포 단계까지 배양한 배아를 이식하는 방법. 일반적으로 성공률이 가장 높습니다.
#   ICSI                : 정자를 난자에 직접 주입하는 미세주입술.
#   IVF                 : 표준 시험관 시술.
#   IUI                 : 인공수정. 자궁 내로 정자를 주입하는 방식이며 정의상 본인 난자를 사용합니다.

def classify_tx(x):
    t = str(x).upper().strip()
    if 'BLASTOCYST' in t: return 'Blastocyst_Transfer'
    if 'ICSI' in t:       return 'ICSI'
    if 'IVF' in t:        return 'IVF'
    if 'IUI' in t:        return 'IUI'
    return 'Unknown'

eda_df['시술_분류_그룹'] = eda_df['특정 시술 유형'].fillna('').apply(classify_tx)

# IUI는 정의상 본인의 난자를 사용하므로, 이 그룹의 '난자 출처'를 일괄로 본인 제공으로 정리합니다.
# 일부 행에서 난자 출처가 누락되어 있을 때 이 보정이 시각화에서 도움이 됩니다.
eda_df.loc[eda_df['시술_분류_그룹'] == 'IUI', '난자 출처'] = '본인 제공'

print('EDA 전용 DataFrame이 준비되었습니다.')
print()
print('시술 분류별 건수')
print(eda_df['시술_분류_그룹'].value_counts())

In [ ]:
# ============================================================
# 타겟 클래스 분포 확인
# ============================================================
# 분류 문제에서 가장 먼저 봐야 할 것은 클래스 비율입니다.
# 한 쪽으로 크게 치우친 데이터에서는 모델이 다수 클래스만 예측해도
# 정확도가 높아 보이는 함정이 발생하기 때문입니다.

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# (1) 절대 건수 막대 그래프
counts = df_raw[TARGET].value_counts()
bars = axes[0].bar(['실패 (0)', '성공 (1)'], counts.values,
                   color=['#B3D4FF', '#FFB3B3'], edgecolor='gray', linewidth=0.5)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'{val:,}건\n({val/len(df_raw)*100:.1f}%)',
                 ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_title('타겟 클래스 분포', fontsize=13)
axes[0].set_ylabel('건수')
axes[0].set_ylim(0, max(counts.values) * 1.2)

# (2) 비율 파이 차트
axes[1].pie(counts.values, labels=['실패 (0)', '성공 (1)'],
            colors=['#B3D4FF', '#FFB3B3'], autopct='%1.1f%%',
            startangle=90, pctdistance=0.75,
            wedgeprops=dict(alpha=0.8, edgecolor='white', linewidth=2))
axes[1].set_title('클래스 비율', fontsize=13)

plt.suptitle(f'임신 성공 여부 분포 (전체 {len(df_raw):,}건)', fontsize=14)
plt.tight_layout()
plt.show()

# scale_pos_weight는 음성 클래스 건수를 양성 클래스 건수로 나눈 값입니다.
# LightGBM과 CatBoost에 이 값을 전달하면 양성 샘플의 손실에 가중치가 곱해져,
# 불균형한 데이터에서도 모델이 소수 클래스의 패턴을 잊지 않도록 도와줍니다.
scale_pos_weight = (df_raw[TARGET]==0).sum() / (df_raw[TARGET]==1).sum()
print(f'클래스 불균형 비율 (scale_pos_weight): {scale_pos_weight:.3f}')
print('실패가 성공보다 약 2.87배 많은 구조이며, 모델 학습 시 보정이 필요합니다.')

In [ ]:
# ============================================================
# 결측치 점검
# ============================================================
# 어느 컬럼에 얼마나 많은 결측이 있는지 확인합니다.
# 결측률이 95% 이상인 컬럼은 사실상 정보가 없는 것으로 보고 제거 후보로 분류합니다.
# 그보다 낮은 결측은 도메인에 맞는 적절한 값으로 채우거나
# 결측 자체를 신호로 표현하는 별도 플래그를 만들 수도 있습니다.

missing = df_raw.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df_raw) * 100).round(1)
missing_df = pd.DataFrame({'결측 건수': missing, '결측률(%)': missing_pct})
missing_df = missing_df[missing_df['결측 건수'] > 0]

print(f'결측치가 존재하는 컬럼은 총 {len(missing_df)}개입니다.')
print()
print(missing_df.to_string())

In [ ]:
# ============================================================
# 연령대별 임신 성공률 분석
# ============================================================
# 임상적으로 가장 잘 알려진 사실 가운데 하나가 '나이가 많을수록 임신 성공률이 떨어진다'입니다.
# 다만 만 45세 이상 구간에서는 기증 난자 사용 비율이 높아져,
# 자가 난자 기준의 임신 성공률 하락 추세가 일부 역전되어 보일 수 있습니다.
# 이 점을 함께 확인하기 위해 세 가지 시각화를 나란히 배치했습니다.

age_order = ['만18-34세', '만35-37세', '만38-39세', '만40-42세', '만43-44세', '만45-50세']

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# (1) 6단계 연령대별 막대 그래프
age_success = eda_df[eda_df['시술 당시 나이'] != '알 수 없음'].groupby(
    '시술 당시 나이')[TARGET].mean().reindex(age_order)
bars = axes[0].bar(range(len(age_success)), age_success.values,
                   color='steelblue', alpha=0.7)
axes[0].set_xticks(range(len(age_success)))
axes[0].set_xticklabels(age_success.index, rotation=30, ha='right', fontsize=9)
axes[0].set_title('연령대별 임신 성공률', fontsize=13)
axes[0].set_ylabel('임신 성공률')
axes[0].set_ylim(0, max(age_success.values) * 1.25)
axes[0].axhline(y=df_raw[TARGET].mean(), color='red', linestyle='--',
                label=f'전체 평균 {df_raw[TARGET].mean()*100:.1f}%')
axes[0].legend()
for bar, val in zip(bars, age_success.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val*100:.1f}%', ha='center', va='bottom', fontsize=9,
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                           edgecolor='steelblue', linewidth=1))

# (2) 3단계 임상 위험군별 그래프
risk_order = ['정상_임신군', '고위험_임신군', '초고위험_임신군']
risk_success = eda_df[eda_df['임신_위험도_범주'] != '미분류'].groupby(
    '임신_위험도_범주')[TARGET].mean().reindex(risk_order)
risk_colors = {'정상_임신군': 'steelblue', '고위험_임신군': 'orange', '초고위험_임신군': 'red'}
bar_colors = [risk_colors[r] for r in risk_order]
bars2 = axes[1].bar(range(len(risk_success)), risk_success.values,
                    color=bar_colors, alpha=0.7)
axes[1].set_xticks(range(len(risk_success)))
axes[1].set_xticklabels(risk_success.index, rotation=20, ha='right', fontsize=9)
axes[1].set_title('위험도 범주별 임신 성공률', fontsize=13)
axes[1].set_ylabel('임신 성공률')
axes[1].set_ylim(0, max(risk_success.values) * 1.25)
axes[1].axhline(y=df_raw[TARGET].mean(), color='red', linestyle='--',
                label=f'전체 평균 {df_raw[TARGET].mean()*100:.1f}%')
axes[1].legend()
for bar, val in zip(bars2, risk_success.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val*100:.1f}%', ha='center', va='bottom', fontsize=9,
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                           edgecolor='gray', linewidth=1))

# (3) 연령대별 인구 분포
age_counts = eda_df[eda_df['시술 당시 나이'].isin(age_order)].groupby(
    '시술 당시 나이').size().reindex(age_order)
age_colors_pie = ['steelblue', 'orange', 'orange', 'red', 'red', 'red']
axes[2].pie(age_counts.values, labels=age_counts.index,
            colors=age_colors_pie, autopct='%1.1f%%',
            startangle=90, pctdistance=0.75,
            wedgeprops=dict(alpha=0.7, edgecolor='white', linewidth=2))
axes[2].set_title('연령대 분포', fontsize=13)

plt.suptitle('연령대별 분석', fontsize=14)
plt.tight_layout()
plt.show()
print('관찰: 나이가 들수록 임신 성공률이 점진적으로 낮아지며,')
print('만 45-50세 구간에서는 기증 난자 사용이 늘어나 그래프가 소폭 반등합니다.')

In [ ]:
# ============================================================
# 시술 유형별 임신 성공률 분석
# ============================================================
# 같은 환자라도 어떤 시술을 받았는지에 따라 결과가 크게 달라집니다.
# 일반적으로 배반포 단계까지 배양한 배아를 이식하는 Blastocyst Transfer가 성공률이 가장 높고,
# IUI(인공수정)는 자궁 내 환경에서의 자연 수정에 가까워 IVF 계열 대비 성공률이 낮은 편입니다.
# 두 번째 그래프인 히트맵은 나이대와 시술 유형이 만나는 지점의 성공률을 보여 줍니다.

treatment_order = ['Blastocyst_Transfer', 'ICSI', 'IVF', 'IUI']
treatment_colors = {
    'Blastocyst_Transfer': '#C9B8E8',
    'ICSI': '#FFF3B0',
    'IVF': '#B8D8E8',
    'IUI': '#FFD9B0',
}
age_order = ['만18-34세', '만35-37세', '만38-39세', '만40-42세', '만43-44세', '만45-50세']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (1) 시술 유형별 임신 성공률 막대 그래프
tx_success = eda_df[eda_df['시술_분류_그룹'] != 'Unknown'].groupby(
    '시술_분류_그룹')[TARGET].mean().reindex(treatment_order).dropna()
bar_colors = [treatment_colors[t] for t in tx_success.index]
bars = axes[0].bar(range(len(tx_success)), tx_success.values,
                   color=bar_colors, edgecolor='gray', linewidth=0.5)
axes[0].set_xticks(range(len(tx_success)))
axes[0].set_xticklabels(tx_success.index, rotation=20, ha='right', fontsize=10)
axes[0].set_title('시술 유형별 임신 성공률', fontsize=13)
axes[0].set_ylabel('임신 성공률')
axes[0].set_ylim(0, max(tx_success.values) * 1.25)
axes[0].axhline(y=df_raw[TARGET].mean(), color='red', linestyle='--',
                label=f'전체 평균 {df_raw[TARGET].mean()*100:.1f}%')
axes[0].legend()
for bar, val in zip(bars, tx_success.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val*100:.1f}%', ha='center', va='bottom', fontsize=10,
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                           edgecolor='gray', linewidth=1))

# (2) 나이대 × 시술 유형 히트맵
df_filtered = eda_df[
    (eda_df['시술 당시 나이'].isin(age_order)) &
    (eda_df['시술_분류_그룹'].isin(treatment_order))
]
pivot = df_filtered.groupby(['시술 당시 나이', '시술_분류_그룹'])[TARGET].mean().unstack()
pivot = pivot.reindex(age_order)[treatment_order]
pivot_pct = (pivot * 100).round(1)

sns.heatmap(pivot_pct, annot=True, fmt='.1f', cmap='coolwarm',
            vmin=0, vmax=50, linewidths=0.5, linecolor='white',
            annot_kws={'size': 11, 'weight': 'bold'},
            cbar_kws={'label': '임신 성공률 (%)'}, ax=axes[1])
axes[1].set_title('나이대 × 시술 유형별 임신 성공률 (%)', fontsize=13)
axes[1].set_xlabel('시술 유형', fontsize=10)
axes[1].set_ylabel('나이대', fontsize=10)
axes[1].set_yticklabels(axes[1].get_yticklabels(), rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 이식된 배아 수별 임신 성공률
# ============================================================
# 직관적으로는 배아를 여러 개 이식할수록 성공률이 높아질 것 같지만,
# 데이터에서는 오히려 1개를 이식할 때 성공률이 가장 높게 나타납니다.
#
# 이는 임상 현장의 의사 결정 패턴과 일치합니다.
# 배아의 질이 양호하면 한 개만 이식해도 충분하다고 판단해 단일 배아 이식을 선택하고,
# 반대로 배아의 질이 떨어지면 성공 가능성을 높이기 위해 여러 개를 이식합니다.
# 따라서 '이식된 배아 수가 적다'는 것은 '좋은 배아가 있어 이식 수를 줄였다'는 신호로 해석됩니다.

emb_data = eda_df.groupby('이식된 배아 수')[TARGET].agg(['mean', 'count']).reset_index()
emb_data.columns = ['이식수', '성공률', '샘플수']
emb_data = emb_data[emb_data['이식수'] > 0]   # 0개는 시술이 이식 단계까지 가지 않은 케이스라 제외합니다.

# 평균보다 높은 그룹은 진한 초록, 평균 이하는 회색으로 표시해 가독성을 높였습니다.
colors = ['#B4B2A9' if v < df_raw[TARGET].mean() else '#1D9E75'
          for v in emb_data['성공률']]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(emb_data['이식수'].astype(int), emb_data['성공률'] * 100,
              color=colors, width=0.55, zorder=3)
ax.axhline(df_raw[TARGET].mean()*100, color='#E24B4A', lw=1.5, ls='--',
           zorder=4, label=f'전체 평균 {df_raw[TARGET].mean()*100:.1f}%')

for bar, (_, row) in zip(bars, emb_data.iterrows()):
    ax.annotate(
        f"{row['성공률']*100:.1f}%\n(n={int(row['샘플수']):,})",
        xy=(bar.get_x() + bar.get_width()/2, row['성공률']*100),
        xytext=(0, 7), textcoords='offset points',
        ha='center', va='bottom', fontsize=9, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='#aaa', lw=1)
    )
ax.plot(emb_data['이식수'].astype(int), emb_data['성공률']*100,
        'o--', color='#534AB7', lw=1.8, ms=6, zorder=5, label='추세선')

ax.set_xlabel('이식된 배아 수 (개)', fontsize=12)
ax.set_ylabel('임신 성공률 (%)', fontsize=12)
ax.set_xticks(emb_data['이식수'].astype(int))
ax.set_ylim(0, 45)
ax.yaxis.grid(True, linestyle=':', color='#D3D1C7', zorder=0)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(fontsize=10)
ax.set_title('이식된 배아 수별 임신 성공률', fontsize=13)
plt.tight_layout()
plt.show()
print('관찰: 한 개의 배아만 이식한 경우의 임신 성공률이 가장 높습니다.')
print('해석: 배아의 질이 좋을 때 단일 이식을 선택하는 임상 패턴이 반영된 결과로 볼 수 있습니다.')

In [ ]:
# ============================================================
# 동결 배아 사용 여부와 기증 난자 사용 비율 분석
# ============================================================
# 만 45세 이상 구간에서 임신 성공률이 살짝 올라간 것처럼 보이는 진짜 이유는
# 그 연령대에서 기증 난자 사용 비율이 높기 때문입니다.
# 자가 난자가 아닌 기증 난자(보통 더 젊은 여성으로부터 받은 난자)를 사용하면
# 산모의 나이와 무관하게 난자 자체의 질이 높으므로 성공률이 유지되거나 오히려 높아집니다.
#
# 동결 배아 역시 비슷한 맥락에서 보아야 합니다.
# 신선 배아는 채취-수정-이식이 같은 주기에 이루어지지만,
# 동결 배아는 자궁 내막 상태가 좋은 별도의 주기에 이식되는 경우가 많아
# 고령 산모에게 더 유리한 환경이 만들어질 수 있습니다.

age_order = ['만18-34세', '만35-37세', '만38-39세', '만40-42세', '만43-44세', '만45-50세']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (1) 나이대별 동결 배아 사용 여부 성공률
pivot_frozen = eda_df[eda_df['시술 당시 나이'].isin(age_order)].groupby(
    ['시술 당시 나이', '동결 배아 사용 여부'])[TARGET].mean().unstack() * 100
pivot_frozen = pivot_frozen.reindex(age_order)

x = np.arange(len(age_order))
width = 0.35
bars1 = axes[0].bar(x - width/2, pivot_frozen[0], width,
                    label='동결 미사용', color='#B3D4FF', edgecolor='gray', linewidth=0.5)
bars2 = axes[0].bar(x + width/2, pivot_frozen[1], width,
                    label='동결 사용', color='#FFB3B3', edgecolor='gray', linewidth=0.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(age_order, rotation=30, ha='right', fontsize=9)
axes[0].set_title('나이대별 동결 배아 사용 여부 임신 성공률', fontsize=11)
axes[0].set_ylabel('임신 성공률 (%)')
axes[0].set_ylim(0, 45)
axes[0].axhline(y=df_raw[TARGET].mean()*100, color='red', linestyle='--',
                label=f'전체 평균 {df_raw[TARGET].mean()*100:.1f}%')
axes[0].legend(fontsize=9)
for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    if h > 0:
        axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.5,
                     f'{h:.1f}%', ha='center', va='bottom', fontsize=8,
                     bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                               edgecolor='gray', linewidth=0.8))

# (2) 나이대별 기증 난자 사용 비율
donor_rate = eda_df[eda_df['시술 당시 나이'].isin(age_order)].groupby(
    '시술 당시 나이').apply(
    lambda g: (g['난자 출처'] == '기증 제공').mean() * 100
).reindex(age_order)
bars3 = axes[1].bar(range(len(age_order)), donor_rate.values,
                    color='#C9B8E8', edgecolor='gray', linewidth=0.5)
axes[1].set_xticks(range(len(age_order)))
axes[1].set_xticklabels(age_order, rotation=30, ha='right', fontsize=9)
axes[1].set_title('나이대별 기증 난자 사용 비율', fontsize=11)
axes[1].set_ylabel('기증 난자 사용 비율 (%)')
axes[1].set_ylim(0, max(donor_rate.values) * 1.3)
for bar, val in zip(bars3, donor_rate.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.3,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()
print('관찰: 만 40세 이상에서 동결 배아 사용 시 신선 배아보다 성공률이 높은 역전 현상이 나타납니다.')
print('해석: 고령일수록 기증 난자 사용 비율이 함께 증가하며, 만 45-50세 구간의 성공률 회복은')
print('이 두 요인이 결합한 결과로 볼 수 있습니다.')

In [ ]:
# ============================================================
# 시술 / 임신 / 출산 횟수별 임신 성공률
# ============================================================
# 횟수 컬럼은 '0회', '1회', ..., '6회 이상' 같은 문자열 형태로 저장되어 있습니다.
# 정규식을 사용해 첫 번째 숫자만 추출하고, 시각화에서는 5회 이상을 5로 묶어 단순화합니다.
# 모델 학습 단계에서는 클립 없이 정수 그대로 사용합니다(셀 후반부의 parse_count 함수).

def parse_count_eda(series):
    """
    문자열 형태의 횟수 컬럼을 정수로 변환합니다 (시각화 전용).
      str.extract(r'(\d+)') : 첫 번째 숫자 그룹을 추출합니다.
      fillna(0)              : 추출에 실패한 행은 0으로 처리합니다.
      clip(upper=5)          : 시각화 단순화를 위해 5회 이상은 5로 묶습니다.
    """
    return series.astype(str).str.extract(r'(\d+)')[0].fillna(0).astype(int).clip(upper=5)

cycle_cols_eda = {
    '총 시술 횟수': '총 시술 횟수',
    'IVF 시술 횟수': 'IVF 시술 횟수',
    '총 임신 횟수': '총 임신 횟수',
    '총 출산 횟수': '총 출산 횟수',
}

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (col_orig, col_label) in zip(axes, cycle_cols_eda.items()):
    parsed = parse_count_eda(eda_df[col_orig])
    success_by_count = eda_df.groupby(parsed)[TARGET].mean() * 100
    count_by_group   = eda_df.groupby(parsed).size()
    bars = ax.bar(success_by_count.index, success_by_count.values,
                  color='#B3D4FF', edgecolor='gray', linewidth=0.5)
    ax.set_title(col_label, fontsize=10)
    ax.set_xlabel('횟수 (5=5회 이상)', fontsize=8)
    ax.set_ylabel('성공률 (%)', fontsize=8)
    ax.set_ylim(0, max(success_by_count.values) * 1.3)
    ax.axhline(y=df_raw[TARGET].mean()*100, color='red', linestyle='--', linewidth=1)
    for bar, val, cnt in zip(bars, success_by_count.values, count_by_group.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}%\n({cnt:,}건)', ha='center', va='bottom', fontsize=7)

plt.suptitle('시술 / 임신 / 출산 횟수별 임신 성공률', fontsize=13)
plt.tight_layout()
plt.show()
print('관찰: 모든 횟수 변수에서 0회(첫 시술 또는 첫 임신)일 때 성공률이 가장 높습니다.')
print('해석: 횟수가 누적될수록 더 까다로운 케이스가 모이는 경향이 반영된 결과로 볼 수 있습니다.')

In [ ]:
# ============================================================
# (추가 EDA 1) 수치형 변수의 이상치 분포
# ============================================================
# 난자/배아 관련 12개 수치형 변수의 분포를 박스플롯으로 살펴봅니다.
# 모두 0에 가까운 값이 다수이고 일부 행에서만 매우 큰 값이 나타나는 우편향(skewed) 구조입니다.
# 이러한 분포에서는 평균값이 중앙값에서 크게 벗어나 통계 해석이 왜곡되며,
# 모델 입장에서도 극단값이 학습에 과도한 영향을 미칠 수 있습니다.
#
# 발견: 12개 변수 중 일부는 분포의 99% 분위수 이상에서 매우 긴 꼬리를 가집니다.
# 이러한 변수는 다음 셀에서 살펴볼 로그 변환의 후보가 됩니다.

outlier_cols = [
    '총 생성 배아 수', '미세주입된 난자 수', '미세주입에서 생성된 배아 수',
    '저장된 배아 수', '미세주입 후 저장된 배아 수', '해동된 배아 수',
    '해동 난자 수', '수집된 신선 난자 수', '저장된 신선 난자 수',
    '혼합된 난자 수', '파트너 정자와 혼합된 난자 수', '기증자 정자와 혼합된 난자 수'
]

# 우편향이 가장 심한 3개 변수를 사전에 확인하여 강조 표시 대상으로 지정합니다.
# 비대칭도(skewness)가 큰 순서대로 상위 3개를 선택합니다.
skewness = eda_df[outlier_cols].skew().sort_values(ascending=False)
top3_skewed = skewness.head(3).index.tolist()
print('우편향이 가장 심한 3개 변수 (skewness 기준)')
for col in top3_skewed:
    print(f'  {col}: skewness={skewness[col]:.2f}')
print()

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(outlier_cols):
    is_top3 = col in top3_skewed
    
    # 강조 대상은 빨간 테두리, 일반은 회색 테두리
    box_color = '#FFB3B3' if is_top3 else 'steelblue'
    edge_color = '#D62728' if is_top3 else 'gray'
    edge_width = 2.0 if is_top3 else 0.8
    
    bp = axes[i].boxplot(
        eda_df[col].dropna(), vert=True, patch_artist=True,
        boxprops=dict(facecolor=box_color, alpha=0.6, 
                      edgecolor=edge_color, linewidth=edge_width),
        medianprops=dict(color='red', linewidth=2),
    )
    
    title_color = '#D62728' if is_top3 else 'black'
    title_text = f'{col}\n(skew={skewness[col]:.1f})'
    if is_top3:
        title_text = '◆ ' + title_text
    axes[i].set_title(title_text, fontsize=9, color=title_color, 
                       fontweight='bold' if is_top3 else 'normal')
    axes[i].set_ylabel('값', fontsize=8)
    
    # 99% 분위수 기준선 표시 (이상치 경계)
    p99 = eda_df[col].quantile(0.99)
    if not pd.isna(p99):
        axes[i].axhline(y=p99, color='orange', linestyle=':', linewidth=1, alpha=0.7)

plt.suptitle('수치형 변수의 이상치 분포  (◆ 표시: 우편향이 가장 심한 3개 변수)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print()
print('관찰: 모든 변수에서 박스(IQR)는 매우 좁고, 일부 점들이 박스의 수십 배 위에 흩어져 있습니다.')
print('이러한 우편향 분포는 다음 셀의 로그 변환을 통해 어느 정도 완화될 수 있습니다.')

In [ ]:
# ============================================================
# (추가 EDA 2) 로그 변환 전후 분포 비교
# ============================================================
# 우편향이 심한 변수에 log(x+1) 변환을 적용하면 분포가 정규에 가까워져
# 통계적 해석이 수월해지고 일부 모델에서는 학습 안정성이 개선됩니다.
# 트리 기반 모델(LightGBM, CatBoost)은 단조 변환에 영향을 받지 않으므로
# 본 노트북의 최종 모델에는 적용되지 않지만, EDA 단계에서 변수 분포를
# 균형 있게 살펴보는 데 도움이 됩니다.
#
# log(x+1)을 사용하는 이유: x=0인 경우 log(0)이 정의되지 않으므로 +1을 더합니다.

# 강조: 변환 효과가 가장 극적인 변수 2개를 위쪽에 크게 배치합니다.
# 이전 셀에서 계산한 skewness 상위 2개를 사용합니다.
top2_skewed = top3_skewed[:2]

# 그림 구성: 위쪽 1행에 강조 변수 2개를 크게, 아래쪽 5행 4열에 나머지 10개를 작게
fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(6, 4, height_ratios=[1.5, 1, 1, 1, 1, 1], hspace=0.55, wspace=0.3)

# ------------------------------------------------------------
# 위쪽: 강조 변수 2개 (각각 원본/로그 한 쌍)
# ------------------------------------------------------------
for i, col in enumerate(top2_skewed):
    # 원본
    ax_orig = fig.add_subplot(gs[0, i*2])
    ax_orig.boxplot(eda_df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='#FFB3B3', alpha=0.7,
                                  edgecolor='#D62728', linewidth=1.5),
                    medianprops=dict(color='red', linewidth=2))
    ax_orig.set_title(f'◆ {col}\n(원본 — skew={eda_df[col].skew():.2f})',
                       fontsize=10, color='#D62728', fontweight='bold')
    
    # 로그 변환
    ax_log = fig.add_subplot(gs[0, i*2 + 1])
    log_val = np.log1p(eda_df[col].dropna())
    ax_log.boxplot(log_val, patch_artist=True,
                   boxprops=dict(facecolor='#B3E8B3', alpha=0.7,
                                 edgecolor='#1D9E75', linewidth=1.5),
                   medianprops=dict(color='red', linewidth=2))
    ax_log.set_title(f'{col}\n(log 변환 — skew={log_val.skew():.2f})',
                      fontsize=10, color='#1D9E75', fontweight='bold')

# ------------------------------------------------------------
# 아래쪽: 나머지 변수 (압축 표시)
# ------------------------------------------------------------
remaining = [c for c in outlier_cols if c not in top2_skewed]
for i, col in enumerate(remaining):
    row = (i // 2) + 1
    col_pos = (i % 2) * 2
    
    ax1 = fig.add_subplot(gs[row, col_pos])
    ax1.boxplot(eda_df[col].dropna(), patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.5))
    ax1.set_title(f'{col[:15]}...\n(원본)' if len(col) > 15 else f'{col}\n(원본)',
                   fontsize=8)
    
    ax2 = fig.add_subplot(gs[row, col_pos + 1])
    ax2.boxplot(np.log1p(eda_df[col].dropna()), patch_artist=True,
                boxprops=dict(facecolor='coral', alpha=0.5))
    ax2.set_title(f'{col[:15]}...\n(log)' if len(col) > 15 else f'{col}\n(log)',
                   fontsize=8)

plt.suptitle('로그 변환 전후 분포 비교  (◆ 표시: 변환 효과가 가장 큰 2개 변수)',
             fontsize=14, fontweight='bold', y=0.995)
plt.show()
print()
print('관찰: 로그 변환 후 박스플롯의 박스(IQR)가 그림 안에서 차지하는 비율이 늘어나')
print('전체 데이터의 분포 형태를 한눈에 파악하기 수월해집니다.')
print('해석: 트리 기반 모델은 단조 변환에 영향을 받지 않으므로 본 모델은 원본 그대로 사용하나,')
print('선형 모델이나 신경망을 사용할 때는 이 변환이 의미 있는 차이를 만들 수 있습니다.')

In [ ]:
# ============================================================
# (추가 EDA 3) 난자 출처별 임신 성공률 — 자가 vs 기증 [핵심 발견]
# ============================================================
# 기본 EDA(셀 11)에서 만 45-50세 구간의 임신 성공률 회복 현상을 확인했습니다.
# 그 진짜 원인이 '기증 난자 사용 비율 증가'에 있다는 점을 데이터로 직접 확인합니다.
#
# 두 개의 히트맵을 나란히 배치하여 자가 난자 사용 그룹과 기증 난자 사용 그룹의
# 임신 성공률을 분리해서 보여 줍니다. 만 45-50세 구간을 보시면,
# 자가 난자에서는 매우 낮은 성공률이, 기증 난자에서는 그보다 훨씬 높은 성공률이 나타납니다.
# 이 데이터가 만 45-50세 구간 평균을 끌어올린 것입니다.

age_order = ['만18-34세', '만35-37세', '만38-39세', '만40-42세', '만43-44세', '만45-50세']
treatment_order = ['Blastocyst_Transfer', 'ICSI', 'IVF', 'IUI']

fig, axes = plt.subplots(1, 2, figsize=(22, 7))

for ax, source, title in zip(
    axes,
    ['본인 제공', '기증 제공'],
    ['본인 난자 (자가)', '기증 난자']
):
    df_source = eda_df[
        (eda_df['시술 당시 나이'].isin(age_order)) &
        (eda_df['시술_분류_그룹'].isin(treatment_order)) &
        (eda_df['난자 출처'] == source)
    ]

    pivot = df_source.groupby(['시술 당시 나이', '시술_분류_그룹'])[TARGET].mean().unstack()
    pivot = pivot.reindex(age_order).reindex(columns=treatment_order)
    pivot_pct = (pivot * 100).round(1)
    mask = pivot_pct.isna()

    sns.heatmap(
        pivot_pct, annot=True, fmt='.1f', cmap='coolwarm',
        vmin=0, vmax=50, linewidths=0.5, linecolor='white',
        annot_kws={'size': 11, 'weight': 'bold'},
        cbar_kws={'label': '임신 성공률 (%)'},
        mask=mask, ax=ax
    )

    # 데이터 없음 표시
    for i in range(len(age_order)):
        for j in range(len(treatment_order)):
            if mask.iloc[i, j]:
                ax.text(j + 0.5, i + 0.5, '데이터\n없음',
                        ha='center', va='center', fontsize=8,
                        color='gray',
                        bbox=dict(boxstyle='round,pad=0.3',
                                  facecolor='#F0F0F0', edgecolor='lightgray'))

    # ★ 핵심 강조: 만 45-50세 행에 노란 박스
    from matplotlib.patches import Rectangle
    rect = Rectangle((0, 5), len(treatment_order), 1,
                     fill=False, edgecolor='#FFA500', linewidth=4, zorder=10)
    ax.add_patch(rect)
    
    ax.set_title(f'{title}', fontsize=13, fontweight='bold')
    ax.set_xlabel('시술 유형', fontsize=10)
    ax.set_ylabel('나이대', fontsize=10)
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=9)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=9, rotation=0)

# 강조 안내 텍스트 (그림 사이)
fig.text(0.5, 0.02,
         '주황색 박스 (만 45-50세): 자가 난자에서는 성공률이 낮으나, 기증 난자에서는 회복되는 모습을 보입니다.',
         ha='center', fontsize=11, color='#D97706', fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#FFF8E7', 
                   edgecolor='#FFA500', linewidth=1.5))

plt.suptitle('나이대 × 시술 유형별 임신 성공률 — 난자 출처에 따른 비교',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.subplots_adjust(bottom=0.13)
plt.show()

# 만 45-50세의 두 그룹 성공률을 출력하여 메시지를 정량화합니다.
own_4550 = eda_df[
    (eda_df['시술 당시 나이'] == '만45-50세') & 
    (eda_df['난자 출처'] == '본인 제공')
][TARGET].mean() * 100
donor_4550 = eda_df[
    (eda_df['시술 당시 나이'] == '만45-50세') & 
    (eda_df['난자 출처'] == '기증 제공')
][TARGET].mean() * 100

print()
print('만 45-50세 그룹의 임신 성공률 비교')
print(f'  자가 난자 사용: {own_4550:.1f}%')
print(f'  기증 난자 사용: {donor_4550:.1f}%')
print(f'  차이:           {donor_4550 - own_4550:+.1f}%p')
print()
print('이 차이가 만 45-50세 구간 평균 성공률을 끌어올린 진짜 원인입니다.')
print('모델 학습 시 자가난자×나이, 기증난자×나이 변수를 분리하여 만든 까닭이기도 합니다.')

In [ ]:
# ============================================================
# (추가 EDA 4) 불임 원인별 임신 성공률 비교
# ============================================================
# 환자가 가지고 있는 불임 원인의 종류에 따라 시술 성공률이 달라지는지 살펴봅니다.
# 각 원인 컬럼은 0/1 이진 변수로, '있음 vs 없음' 두 그룹의 성공률을 비교합니다.
# 격차가 클수록 그 원인이 결과에 더 직접적인 영향을 미친다고 볼 수 있습니다.

cause_cols = [
    '남성 주 불임 원인', '여성 주 불임 원인',
    '부부 주 불임 원인', '부부 부 불임 원인',
    '남성 부 불임 원인', '여성 부 불임 원인',
    '불임 원인 - 난관 질환', '불임 원인 - 남성 요인',
    '불임 원인 - 배란 장애', '불임 원인 - 자궁내막증',
    '불임 원인 - 자궁내막 또는 자궁 내 유착',
    '불명확 불임 원인'
]
# 데이터에 실제 존재하는 컬럼만 선택
cause_cols = [c for c in cause_cols if c in eda_df.columns]

# 각 원인별로 '있음', '없음' 그룹의 성공률 계산
success_rates = {}
for col in cause_cols:
    success_rates[col] = {
        '있음': eda_df[eda_df[col] == 1][TARGET].mean(),
        '없음': eda_df[eda_df[col] == 0][TARGET].mean(),
    }

# 격차(있음 - 없음의 차이)가 큰 순으로 정렬하면 메시지가 더 잘 보입니다.
gap = {c: success_rates[c]['있음'] - success_rates[c]['없음'] for c in cause_cols}
sorted_cols = sorted(cause_cols, key=lambda c: abs(gap[c]), reverse=True)

# 가장 큰 격차를 보이는 변수 (강조 대상)
top_cause = sorted_cols[0]
print(f'성공률 격차가 가장 큰 불임 원인: {top_cause}')
print(f'  있음: {success_rates[top_cause]["있음"]*100:.1f}%')
print(f'  없음: {success_rates[top_cause]["없음"]*100:.1f}%')
print(f'  차이: {gap[top_cause]*100:+.1f}%p')
print()

fig, ax = plt.subplots(figsize=(14, 6.5))

cols = sorted_cols
있음 = [success_rates[c]['있음'] * 100 for c in cols]
없음 = [success_rates[c]['없음'] * 100 for c in cols]

x = np.arange(len(cols))
width = 0.35

# 강조: 격차가 가장 큰 원인의 막대를 진한 색으로 표시
top_idx = 0  # 정렬되어 있으므로 첫 번째가 가장 큰 격차
bar_colors_있음 = ['#D62728' if i == top_idx else '#FFB3B3' for i in range(len(cols))]
bar_colors_없음 = ['#1F77B4' if i == top_idx else '#B3D4FF' for i in range(len(cols))]

bars1 = ax.bar(x - width/2, 있음, width, label='원인 있음',
                color=bar_colors_있음, edgecolor='gray', linewidth=0.5)
bars2 = ax.bar(x + width/2, 없음, width, label='원인 없음',
                color=bar_colors_없음, edgecolor='gray', linewidth=0.5)

# 평균선
mean_rate = eda_df[TARGET].mean() * 100
ax.axhline(y=mean_rate, color='red', linestyle='--', linewidth=1.5,
           label=f'전체 평균 임신 성공률 {mean_rate:.1f}%')

ax.set_xticks(x)
labels = [c.replace('불임 원인 - ', '').replace('불임 원인', '') for c in cols]
ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('임신 성공률 (%)')
ax.set_ylim(0, 35)
ax.set_title('불임 원인별 임신 성공률 (있음 vs 없음)  —  격차 큰 순으로 정렬',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)

# 막대 위 수치 표시
for bar in bars1:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.3,
                f'{h:.1f}%', ha='center', va='bottom', fontsize=8,
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                          edgecolor='#FFB3B3', linewidth=0.8))
for bar in bars2:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.3,
                f'{h:.1f}%', ha='center', va='bottom', fontsize=8,
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                          edgecolor='#B3D4FF', linewidth=0.8))

# 강조 화살표 + 설명 박스
ax.annotate(
    f'가장 큰 격차\n({gap[top_cause]*100:+.1f}%p)',
    xy=(0, max(있음[0], 없음[0]) + 1.5),
    xytext=(2, 32),
    fontsize=10, fontweight='bold', color='#D62728',
    ha='center',
    arrowprops=dict(arrowstyle='->', color='#D62728', lw=1.5),
    bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFF8E7',
              edgecolor='#D62728', linewidth=1.5)
)

plt.tight_layout()
plt.show()
print()
print('관찰: 원인의 종류에 따라 성공률 격차의 크기와 방향이 다릅니다.')
print('해석: 단순히 "불임 원인이 있다"가 아니라, 어떤 원인인지가 결과에 더 중요합니다.')
print('이러한 다양한 신호를 모델은 트리의 분기를 통해 자체적으로 학습합니다.')

In [ ]:
# ============================================================
# (추가 EDA 5) 배아/난자 수치(log) 분포 — 성공 그룹 vs 실패 그룹
# ============================================================
# 앞서 살펴본 수치형 변수들이 임신 성공 여부에 따라 분포가 달라지는지 확인합니다.
# 두 그룹의 박스플롯 중앙값이 명확히 다르다면 그 변수는 결과를 예측하는 신호를 담고 있습니다.
# 로그 변환된 값을 사용하는 이유는 우편향이 심해 원본으로는 분포 차이가 잘 보이지 않기 때문입니다.

# 분석할 변수 6개 (앞서 본 12개 중 핵심만 선택)
log_cols = [
    '총 생성 배아 수', '미세주입에서 생성된 배아 수', '저장된 배아 수',
    '수집된 신선 난자 수', '혼합된 난자 수', '파트너 정자와 혼합된 난자 수'
]

# 임시로 로그 변환 컬럼 생성 (시각화 전용, eda_df에 추가)
for col in log_cols:
    eda_df[col + '_log'] = np.log1p(eda_df[col].fillna(0))

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

# 각 변수에서 성공/실패 그룹 중앙값 차이를 사전 계산
median_diffs = []
for col in log_cols:
    log_col = col + '_log'
    med_fail = eda_df[eda_df[TARGET] == 0][log_col].median()
    med_success = eda_df[eda_df[TARGET] == 1][log_col].median()
    median_diffs.append((col, med_success - med_fail))

# 격차가 가장 큰 변수를 강조 대상으로 지정
median_diffs.sort(key=lambda x: abs(x[1]), reverse=True)
top_var = median_diffs[0][0]
print(f'성공 그룹과 실패 그룹의 중앙값 차이가 가장 큰 변수: {top_var}')
print(f'  차이: {median_diffs[0][1]:+.3f} (log 단위)')
print()

for i, col in enumerate(log_cols):
    log_col = col + '_log'
    is_top = col == top_var
    
    # 성공/실패 그룹 분리
    data_fail = eda_df[eda_df[TARGET] == 0][log_col].dropna()
    data_success = eda_df[eda_df[TARGET] == 1][log_col].dropna()
    
    bp = axes[i].boxplot(
        [data_fail, data_success],
        labels=['실패 (0)', '성공 (1)'],
        patch_artist=True,
        widths=0.5,
        boxprops=dict(facecolor='#B3D4FF', alpha=0.7),
        medianprops=dict(color='red', linewidth=2.5),
    )
    # 성공 그룹 박스만 색상 변경
    bp['boxes'][1].set_facecolor('#FFB3B3')
    
    # 강조 변수의 외곽선 두껍게
    if is_top:
        for box in bp['boxes']:
            box.set_edgecolor('#D62728')
            box.set_linewidth(2)
    
    title_text = f'◆ {col}' if is_top else col
    title_color = '#D62728' if is_top else 'black'
    axes[i].set_title(title_text, fontsize=10, color=title_color,
                       fontweight='bold' if is_top else 'normal')
    axes[i].set_ylabel('log 변환값', fontsize=9)
    axes[i].set_xlabel('임신 성공 여부', fontsize=9)
    
    # 중앙값 차이를 텍스트로 표시
    med_diff = data_success.median() - data_fail.median()
    color_diff = '#D62728' if med_diff > 0 else '#1F77B4'
    axes[i].text(0.5, 0.95,
                  f'중앙값 차이: {med_diff:+.2f}',
                  transform=axes[i].transAxes,
                  ha='center', va='top', fontsize=9, fontweight='bold',
                  color=color_diff,
                  bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                            edgecolor=color_diff, linewidth=1))

plt.suptitle('배아/난자 수치(log) 분포 — 성공 그룹 vs 실패 그룹',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 임시 컬럼 정리
eda_df.drop(columns=[c + '_log' for c in log_cols], inplace=True)

print()
print('관찰: 성공 그룹의 중앙값이 실패 그룹보다 일관되게 위쪽에 위치합니다.')
print('해석: 배아와 난자가 더 많이 생성되었다는 것은 그 자체로 좋은 시술 환경을 의미합니다.')
print('이 신호들은 로그 변환을 거치지 않은 원본 그대로도 트리 모델이 충분히 학습할 수 있습니다.')

In [ ]:
# ============================================================
# (추가 EDA 6) 배아 종류별 임신 성공률 — 해동/동결/신선
# ============================================================
# 배아의 처리 방식에 따라 임신 성공률이 달라지는지 살펴봅니다.
# 임상 현장에서 자궁 내막 상태가 좋은 별도 주기에 동결 배아를 이식하는 경우가 늘고 있는데,
# 이 데이터에서도 그러한 임상적 변화가 결과로 드러나는지 확인합니다.

mean_rate = eda_df[TARGET].mean() * 100

# 강조 대상 사전 계산: 평균 대비 격차가 가장 큰 그룹
groups = ['배아해동_수행', '동결 배아 사용 여부', '신선 배아 사용 여부']
group_labels = ['배아 해동 수행', '동결 배아 사용', '신선 배아 사용']

# 시각화에 필요한 컬럼이 eda_df에 있는지 확인
# 배아해동_수행은 원본 컬럼이 아니므로 임시로 생성합니다.
if '배아해동_수행' not in eda_df.columns:
    if '해동된 배아 수' in eda_df.columns:
        eda_df['배아해동_수행'] = (eda_df['해동된 배아 수'].fillna(0) > 0).astype(int)
    else:
        eda_df['배아해동_수행'] = 0

# 그룹별 성공률
group_rates = []
for col, label in zip(groups, group_labels):
    if col in eda_df.columns:
        rate_0 = eda_df[eda_df[col] == 0][TARGET].mean() * 100
        rate_1 = eda_df[eda_df[col] == 1][TARGET].mean() * 100
        group_rates.append({
            'col': col, 'label': label,
            'rate_0': rate_0, 'rate_1': rate_1,
            'gap': abs(rate_1 - rate_0)
        })

# 격차가 가장 큰 그룹을 강조 대상으로
top_group = max(group_rates, key=lambda x: x['gap'])
print(f'평균 대비 격차가 가장 큰 그룹: {top_group["label"]}')
print(f'  미사용: {top_group["rate_0"]:.1f}% / 사용: {top_group["rate_1"]:.1f}%')
print(f'  차이:   {top_group["rate_1"] - top_group["rate_0"]:+.1f}%p')
print()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, info in enumerate(group_rates):
    is_top = info['col'] == top_group['col']
    
    rates = [info['rate_0'], info['rate_1']]
    labels = ['미사용 (0)', '사용 (1)']
    
    # 강조 대상은 진한 색, 나머지는 연한 색
    if is_top:
        colors = ['#1F77B4', '#D62728']
        edge_width = 1.8
    else:
        colors = ['#B3D4FF', '#FFB3B3']
        edge_width = 0.5
    
    bars = axes[i].bar(labels, rates,
                        color=colors, edgecolor='gray', linewidth=edge_width)
    
    title_text = f'◆ {info["label"]}' if is_top else info['label']
    title_color = '#D62728' if is_top else 'black'
    axes[i].set_title(f'{title_text} 여부별 임신 성공률',
                       fontsize=11, color=title_color,
                       fontweight='bold' if is_top else 'normal')
    axes[i].set_ylabel('임신 성공률 (%)')
    axes[i].set_ylim(0, max(rates) * 1.3 + 5)
    axes[i].axhline(y=mean_rate, color='red', linestyle='--',
                     label=f'전체 평균 {mean_rate:.1f}%')
    axes[i].legend(fontsize=9)
    
    # 막대 위 수치 표시
    for bar, val in zip(bars, rates):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                      f'{val:.1f}%', ha='center', va='bottom', fontsize=11,
                      fontweight='bold' if is_top else 'normal',
                      bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                                edgecolor='gray', linewidth=0.8))
    
    # 강조 대상에는 격차 화살표
    if is_top:
        axes[i].annotate(
            f'{info["rate_1"] - info["rate_0"]:+.1f}%p',
            xy=(0.5, max(rates) + 1.5),
            xytext=(0.5, max(rates) + 4),
            ha='center', fontsize=11, fontweight='bold', color='#D62728',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFF8E7',
                      edgecolor='#D62728', linewidth=1.5)
        )

plt.suptitle('배아 종류별 임신 성공률 비교',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 임시 컬럼 정리
if '배아해동_수행' in eda_df.columns and '해동된 배아 수' in eda_df.columns:
    eda_df.drop(columns=['배아해동_수행'], inplace=True)

print()
print('관찰: 동결 배아 사용 여부가 임신 성공률에 가장 큰 영향을 미치는 것으로 보입니다.')
print('해석: 자궁 내막 상태가 양호한 주기를 골라 이식하는 임상 전략의 효과로 해석됩니다.')

In [ ]:
# ============================================================
# (추가 EDA 7) 나이대와 과거 시술 횟수의 교차 분석 [핵심 발견]
# ============================================================
# 동일한 나이대에서도 과거 시술을 몇 번 받았는지에 따라 임신 성공률이 달라지는지 살펴봅니다.
# 직관적으로는 시술 횟수가 늘수록 성공률이 떨어질 것으로 예상되나,
# 실제 데이터에서는 일부 구간에서 의외의 패턴이 관찰됩니다.

# 시술 횟수를 0회 / 1-2회 / 3회 이상의 세 구간으로 나누어 분석합니다.
# 원본 컬럼이 문자열이라 정수로 파싱한 후 클립을 적용하지 않은 상태로 사용합니다.
def parse_count_eda(s):
    return s.astype(str).str.extract(r'(\d+)')[0].fillna(0).astype(int)

eda_df['총_시술_횟수_int'] = parse_count_eda(eda_df['총 시술 횟수'])
eda_df['시술횟수_구간'] = pd.cut(
    eda_df['총_시술_횟수_int'],
    bins=[-1, 0, 2, 100],
    labels=['0회', '1-2회', '3회 이상']
)

age_order = ['만18-34세', '만35-37세', '만38-39세', '만40-42세', '만43-44세', '만45-50세']

# 나이대 × 시술 횟수 교차표
pivot = eda_df[eda_df['시술 당시 나이'].isin(age_order)].groupby(
    ['시술 당시 나이', '시술횟수_구간'], observed=True
)[TARGET].mean().unstack() * 100
pivot = pivot.reindex(age_order)

# 핵심 발견 사전 계산: 첫 시술(0회) 그룹과 1-2회 그룹의 차이가 큰 나이대 찾기
gaps = pivot['1-2회'] - pivot['0회']
print('나이대별 1-2회 시술 그룹과 0회 시술 그룹의 성공률 차이')
for age in age_order:
    if age in pivot.index:
        diff = gaps[age]
        if not pd.isna(diff):
            arrow = '↑' if diff > 0 else '↓'
            print(f'  {age}: 0회 {pivot.loc[age, "0회"]:.1f}% → 1-2회 {pivot.loc[age, "1-2회"]:.1f}%  ({arrow} {abs(diff):.1f}%p)')
print()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ------------------------------------------------------------
# (왼쪽) 막대 그래프
# ------------------------------------------------------------
x = np.arange(len(age_order))
width = 0.25
colors = ['#B3D4FF', '#FFB3B3', '#B3E8B3']
labels = ['0회', '1-2회', '3회 이상']

for j, (col, color) in enumerate(zip(labels, colors)):
    vals = pivot[col].values
    bars = axes[0].bar(x + (j-1)*width, vals, width,
                        label=col, color=color, edgecolor='gray', linewidth=0.5)
    for bar in bars:
        h = bar.get_height()
        if not pd.isna(h) and h > 0:
            axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.4,
                          f'{h:.1f}%', ha='center', va='bottom', fontsize=8,
                          bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                                    edgecolor=color, linewidth=0.8))

axes[0].set_xticks(x)
axes[0].set_xticklabels(age_order, rotation=30, ha='right', fontsize=9)
axes[0].set_title('나이대 × 시술 횟수 구간별 임신 성공률', fontsize=12, fontweight='bold')
axes[0].set_ylabel('임신 성공률 (%)')
axes[0].set_ylim(0, 50)
axes[0].axhline(y=eda_df[TARGET].mean()*100, color='red', linestyle='--',
                 label=f'전체 평균 {eda_df[TARGET].mean()*100:.1f}%')
axes[0].legend(loc='upper right', fontsize=9)

# ------------------------------------------------------------
# (오른쪽) 히트맵 — 한눈에 보기 쉽게
# ------------------------------------------------------------
sns.heatmap(
    pivot.round(1), annot=True, fmt='.1f', cmap='coolwarm',
    vmin=0, vmax=40, linewidths=0.5, linecolor='white',
    annot_kws={'size': 12, 'weight': 'bold'},
    cbar_kws={'label': '임신 성공률 (%)'},
    ax=axes[1]
)
axes[1].set_title('나이대 × 시술 횟수 구간별 임신 성공률 (히트맵)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('시술 횟수 구간', fontsize=10)
axes[1].set_ylabel('나이대', fontsize=10)
axes[1].set_yticklabels(axes[1].get_yticklabels(), rotation=0, fontsize=10)

# 강조: 가장 큰 양의 격차를 보이는 행에 노란 박스
from matplotlib.patches import Rectangle
positive_gap_ages = gaps[gaps > 0].sort_values(ascending=False)
if len(positive_gap_ages) > 0:
    top_age_for_gap = positive_gap_ages.index[0]
    top_age_idx = age_order.index(top_age_for_gap)
    rect = Rectangle((0, top_age_idx), 3, 1, fill=False,
                     edgecolor='#FFA500', linewidth=4, zorder=10)
    axes[1].add_patch(rect)

plt.suptitle('나이대 × 과거 시술 횟수 교차 분석',
             fontsize=15, fontweight='bold', y=1.00)

# 핵심 발견 텍스트 박스 (그림 아래)
fig.text(0.5, -0.03,
         f'주황색 박스: 0회 시술보다 1-2회 시술 그룹의 성공률이 더 높은 구간이 존재합니다. '
         f'이는 첫 시술에서 실패한 환자들이 재시술 시 더 나은 임상 조건을 갖추기 때문으로 해석됩니다.',
         ha='center', fontsize=11, color='#D97706', fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#FFF8E7',
                   edgecolor='#FFA500', linewidth=1.5))

plt.tight_layout()
plt.subplots_adjust(bottom=0.13)
plt.show()

# 임시 컬럼 정리
eda_df.drop(columns=['총_시술_횟수_int', '시술횟수_구간'], inplace=True, errors='ignore')

print()
print('관찰: 직관과 달리 일부 나이대에서는 과거 1-2회 시술 그룹의 성공률이 첫 시술보다 높습니다.')
print('해석: 첫 시술 실패 후 진단·전처치가 보강되어 두 번째 시도에서 더 좋은 조건을 갖추는')
print('임상 패턴이 반영된 결과로 볼 수 있습니다.')
print('한편 3회 이상 시술 그룹은 모든 나이대에서 성공률이 크게 떨어집니다.')
print('반복 실패가 누적될수록 더 까다로운 케이스가 모이는 영향으로 보입니다.')

In [ ]:
# ============================================================
# (추가 EDA 8) 주요 연속형 변수의 KDE 페어플롯
# ============================================================
# 핵심 연속형 변수들을 한 그림에 모아 변수 간 관계와
# 성공/실패 그룹별 분포 차이를 동시에 살펴봅니다.
# 대각선의 KDE는 각 변수의 단변량 분포를, 비대각선의 KDE는 두 변수의 결합 분포를 나타냅니다.
#
# 색상: 회색 = 실패, 초록 = 성공
# 두 색이 잘 분리될수록 그 변수 쌍이 임신 성공 여부를 잘 구분한다고 해석할 수 있습니다.

# 시각화 전용 컬럼 임시 생성
def parse_count_for_kde(s):
    return s.astype(str).str.extract(r'(\d+)')[0].fillna(0).astype(int).clip(upper=10)

eda_df['_나이_수치_kde'] = eda_df['시술 당시 나이'].map(
    lambda x: age_info_eda.get(x, {'val': np.nan})['val'])
eda_df['_총_시술_kde'] = parse_count_for_kde(eda_df['총 시술 횟수'])
eda_df['_총_임신_kde'] = parse_count_for_kde(eda_df['총 임신 횟수'])
eda_df['_난자수_log_kde'] = np.log1p(eda_df['수집된 신선 난자 수'].fillna(0))

cont_cols = [
    '_나이_수치_kde',
    '이식된 배아 수',
    '_총_시술_kde',
    '_총_임신_kde',
    '_난자수_log_kde',
    TARGET,
]

# 페어플롯은 데이터가 많으면 매우 느려지므로 무작위로 3000개를 샘플링합니다.
# 시각화 전용이며 모델에는 영향을 주지 않습니다.
sample_df = eda_df[cont_cols].dropna().sample(n=min(3000, len(eda_df)), random_state=42)

# 컬럼명을 보기 좋게 변경 (그림 라벨용)
display_names = {
    '_나이_수치_kde': '나이',
    '이식된 배아 수': '이식 배아 수',
    '_총_시술_kde': '총 시술 횟수',
    '_총_임신_kde': '총 임신 횟수',
    '_난자수_log_kde': '난자 수(log)',
    TARGET: '임신 성공',
}
sample_df = sample_df.rename(columns=display_names)

g = sns.pairplot(
    sample_df,
    hue='임신 성공',
    palette={0: '#B4B2A9', 1: '#1D9E75'},
    kind='kde',
    diag_kind='kde',
    plot_kws={'alpha': 0.5, 'fill': True, 'levels': 5},
    diag_kws={'fill': True, 'alpha': 0.5},
    height=2.2,
    aspect=1.0,
)

# 범례 라벨 한글화
if g.legend is not None:
    g.legend.set_title('임신 성공 여부')
    legend_texts = list(g.legend.texts)
    new_labels = ['실패 (0)', '성공 (1)']
    for t, label in zip(legend_texts, new_labels):
        t.set_text(label)

g.fig.suptitle('주요 연속형 변수의 KDE 페어플롯  (3,000건 샘플)',
                fontsize=14, fontweight='bold', y=1.01)

plt.show()

# 임시 컬럼 정리
eda_df.drop(columns=['_나이_수치_kde', '_총_시술_kde', '_총_임신_kde', '_난자수_log_kde'],
             inplace=True, errors='ignore')

print()
print('관찰 포인트')
print('  - 대각선의 단변량 KDE: 두 그룹의 분포 봉우리 위치가 다른 변수가 좋은 신호입니다.')
print('  - 비대각선의 결합 KDE: 두 색의 영역이 분리되어 보일수록 두 변수 조합이 결과를 잘 구분합니다.')
print('  - 나이와 이식 배아 수의 조합에서 두 그룹의 분포가 뚜렷이 어긋난 모습을 확인할 수 있습니다.')

In [ ]:
# ============================================================
# 전처리에 사용할 상수 정의
# ============================================================
# 본 노트북에서 반복적으로 참조하는 값들을 한곳에 모아 정의합니다.
# 이렇게 모아 두면 나중에 정책을 변경할 때 한 곳만 수정하면 되어 유지보수가 수월합니다.

# ------------------------------------------------------------
# 학습에서 제외할 컬럼 목록
# ------------------------------------------------------------
# - 'ID': 행 식별자로 예측에 의미 없음
# - '임신 시도 또는 마지막 임신 경과 연수', '난자 해동 경과일': 결측률이 매우 높아 신호가 거의 없음
# - '불임 원인 - ...' 6종: 값의 대부분이 0인 희소 변수
# - PGS/PGD/착상 전 유전 검사 3종: '유전검사_시행여부' 단일 변수로 통합 사용
COLS_TO_DROP = [
    'ID', '임신 시도 또는 마지막 임신 경과 연수', '난자 해동 경과일',
    '불임 원인 - 여성 요인', '불임 원인 - 자궁경부 문제',
    '불임 원인 - 정자 면역학적 요인', '불임 원인 - 정자 운동성',
    '불임 원인 - 정자 농도', '불임 원인 - 정자 형태',
    'PGS 시술 여부', 'PGD 시술 여부', '착상 전 유전 검사 사용 여부',
]

# ------------------------------------------------------------
# 나이 구간을 대표 수치와 임상 위험군으로 변환하는 매핑
# ------------------------------------------------------------
# 'median'은 구간의 중앙값에 해당하는 대표 나이 수치이고,
# 'risk'는 임상적 위험도 분류입니다.
#
# Normal         : 자연 임신 성공률이 양호한 정상군
# High_Early     : 만 35세 이상, 난소 예비능 저하가 시작되는 시기
# High_Extreme   : 만 40세 이상, 자가 난자 임신 성공률이 급격히 떨어지는 구간
# High_Reversal  : 만 45세 이상, 기증 난자 사용으로 일부 역전이 관찰되는 구간
AGE_INFO = {
    '만18-34세': {'median': 26.0, 'risk': 'Normal'},
    '만35-37세': {'median': 36.0, 'risk': 'High_Early'},
    '만38-39세': {'median': 38.5, 'risk': 'High_Early'},
    '만40-42세': {'median': 41.0, 'risk': 'High_Extreme'},
    '만43-44세': {'median': 43.5, 'risk': 'High_Extreme'},
    '만45-50세': {'median': 47.5, 'risk': 'High_Reversal'},
    '알 수 없음': {'median': None, 'risk': 'Unknown'},
}

# 나이 결측 시 사용할 보정값입니다.
# 38.5는 전체 분포의 중앙값에 가까운 수치로, 통계 학습이 아닌 도메인 판단에 따른 고정값입니다.
AGE_MEDIAN_FILLNA = 38.5

# ------------------------------------------------------------
# 멀티라벨 컬럼 처리에 사용할 알려진 클래스 목록
# ------------------------------------------------------------
# '배아 생성 주요 이유'는 한 행에 여러 값이 콤마로 구분되어 있는 멀티라벨 변수입니다.
# 이 5개 값은 학습 데이터에서 이미 알고 있는 클래스이므로 하드코딩하여
# 평가 데이터에서 새로운 값이 들어오더라도 동일한 5개 컬럼만 생성되도록 보장합니다.
# 이 처리가 데이터 누수 가이드 유형 2를 방지하는 핵심입니다.
MULTILABEL_COL = '배아 생성 주요 이유'
KNOWN_REASONS  = ['현재 시술용', '배아 저장용', '난자 저장용', '기증용', '연구용']

print('전처리 상수 정의가 완료되었습니다.')
print(f'  학습에서 제외할 컬럼: {len(COLS_TO_DROP)}개')
print(f'  나이 구간 매핑: {len(AGE_INFO)}개')
print(f'  멀티라벨 클래스: {len(KNOWN_REASONS)}개')

In [ ]:
# ============================================================
# 전처리 함수 정의
# ============================================================
# 본 노트북의 모든 변수 가공 로직을 네 개의 함수로 모듈화했습니다.
# 학습 데이터와 평가 데이터에 동일한 함수를 동일한 인자로 호출하므로,
# 두 데이터에 적용되는 변환 규칙이 자동으로 일치합니다.
#
#   engineer_features    : 도메인 기반 파생 변수 생성
#   encode_multilabel    : 콤마 구분 멀티라벨을 0/1 컬럼으로 분해
#   preprocess_missing   : 결측치를 고정 상수로 보정
#   build_features       : 위 세 단계를 차례로 호출하는 통합 함수


def engineer_features(df, df_raw_ref=None):
    """
    도메인 지식 기반의 파생 변수들을 생성합니다.
    
    매개변수
        df          : 변환을 적용할 데이터프레임 (이미 COLS_TO_DROP이 적용된 상태)
        df_raw_ref  : 원본 데이터프레임 참조. PGS/PGD 컬럼은 COLS_TO_DROP에서
                      제거되었기 때문에, 유전검사_시행여부 변수를 만들기 위해
                      원본 데이터를 별도로 받아 사용합니다.
                      df_raw_ref와 df는 행 순서가 동일해야 합니다.
    """
    df = df.copy()
    
    # ------------------------------------------------------------
    # 1. 나이 변수 가공
    # ------------------------------------------------------------
    # 원본의 '시술 당시 나이'는 구간 문자열입니다. 모델이 활용하기 좋게
    # 대표 수치(Age_Median)와 임상 위험군(Age_Risk_Group)으로 분리합니다.
    if '시술 당시 나이' in df.columns:
        df['Age_Median'] = df['시술 당시 나이'].map(
            lambda x: AGE_INFO.get(x, {'median': None})['median'])
        df['Age_Risk_Group'] = df['시술 당시 나이'].map(
            lambda x: AGE_INFO.get(x, {'risk': 'Unknown'})['risk'])
        df = df.drop(columns=['시술 당시 나이'])
    
    # ------------------------------------------------------------
    # 2. 유전검사 시행 여부 통합
    # ------------------------------------------------------------
    # PGS와 PGD는 모두 착상 전 유전 진단의 하위 개념입니다.
    # 둘 중 어느 하나라도 시행된 경우 '유전검사_시행여부'를 1로 표시합니다.
    # 원본 컬럼은 COLS_TO_DROP에서 제거되므로 df_raw_ref에서 직접 참조합니다.
    if df_raw_ref is not None:
        df['유전검사_시행여부'] = (
            df_raw_ref['PGS 시술 여부'].notna() |
            df_raw_ref['PGD 시술 여부'].notna()
        ).astype(int).values
    
    # ------------------------------------------------------------
    # 3. 기증 난자 관련 상호작용 변수
    # ------------------------------------------------------------
    # 만 45-50세 구간에서 임신 성공률이 일부 회복되는 현상은 기증 난자 사용 때문입니다.
    # 모델이 '나이가 많은데 성공'한 케이스를 단순히 '나이의 영향이 약하다'로 학습하지 않도록,
    # 자가 난자일 때만 나이 페널티가 작동하는 변수를 별도로 만듭니다.
    #
    #   기증난자×나이 : 기증 난자를 받은 경우에만 값이 살아 있고, 나머지는 0
    #   자가난자×나이 : 자가 난자인 경우에만 값이 살아 있고, 나머지는 0
    #
    # 이렇게 분리하면 모델이 "자가 난자라면 나이가 중요하다"와
    # "기증 난자라면 나이의 영향이 줄어든다"를 모두 학습할 수 있습니다.
    if '난자 출처' in df.columns:
        df['기증난자_여부'] = (df['난자 출처'] == '기증 제공').astype(int)
        age_median = df['Age_Median'].fillna(47.5)   # 결측은 고령으로 보정
        df['기증난자×나이'] = df['기증난자_여부']      * age_median
        df['자가난자×나이'] = (1 - df['기증난자_여부']) * age_median
    
    # ------------------------------------------------------------
    # 4. 배아 관련 파생 변수
    # ------------------------------------------------------------
    # 배아_잉여율 : (총 생성 - 이식)/총 생성. 이식 후 남은 배아 비율
    #              값이 높다는 것은 양질의 배아가 충분히 만들어졌다는 신호로 해석됩니다.
    # Age×배아수  : 나이와 총 배아 수의 곱. 고령에서 배아가 많을 때의 효과를 별도로 표현합니다.
    total    = df.get('총 생성 배아 수', pd.Series(0, index=df.index))
    transfer = df.get('이식된 배아 수',  pd.Series(0, index=df.index))
    df['배아_잉여율']  = np.where(total > 0, (total - transfer) / total, 0).clip(0, 1)
    df['Age×배아수']  = df['Age_Median'].fillna(47.5) * total
    
    # ------------------------------------------------------------
    # 5. 수정률 (정자 품질의 간접 지표)
    # ------------------------------------------------------------
    # 채취된 신선 난자 중 실제로 정자와 혼합되어 수정된 비율입니다.
    # 이 비율이 낮다면 정자 운동성이나 농도에 문제가 있을 가능성이 높습니다.
    collected = df.get('수집된 신선 난자 수', pd.Series(0, index=df.index))
    mixed     = df.get('혼합된 난자 수',      pd.Series(0, index=df.index))
    df['수정률'] = np.where(collected > 0, mixed / collected, 0).clip(0, 1)
    
    # ------------------------------------------------------------
    # 6. 저장 배아 보유 여부 (이진 플래그)
    # ------------------------------------------------------------
    # 저장된 배아가 있다는 것은 이번 주기에서 실패하더라도 다음 시도를 위한 자원이 있다는 뜻입니다.
    # 의외로 단순한 이 플래그가 모델에 유의미한 신호를 제공합니다.
    stored = df.get('저장된 배아 수', pd.Series(0, index=df.index))
    df['저장배아_보유여부'] = (stored > 0).astype(int)
    
    # ------------------------------------------------------------
    # 7. 시술 횟수 통합
    # ------------------------------------------------------------
    # IVF와 DI 시술 횟수를 합쳐 총 시술 횟수를 계산합니다.
    # 횟수 컬럼은 '0회', '1회'와 같은 문자열이므로 정규식으로 숫자만 추출합니다.
    def parse_count(col_name):
        s = df.get(col_name, pd.Series('0회', index=df.index))
        return s.astype(str).str.extract(r'(\d+)')[0].fillna(0).astype(int)
    df['총_시술횟수'] = parse_count('IVF 시술 횟수') + parse_count('DI 시술 횟수')
    
    # ------------------------------------------------------------
    # 8. 기증 정자 사용 여부
    # ------------------------------------------------------------
    # '기증자 정자와 혼합된 난자 수' 컬럼은 기증 정자를 사용하지 않았을 때 결측으로 기록됩니다.
    # 즉 '결측이 아니다'라는 것 자체가 기증 정자를 사용했다는 신호입니다.
    df['기증정자_사용여부'] = df.get(
        '기증자 정자와 혼합된 난자 수',
        pd.Series(np.nan, index=df.index)
    ).notna().astype(int)
    
    return df


def encode_multilabel(df):
    """
    '배아 생성 주요 이유' 컬럼은 한 행에 여러 값이 콤마로 구분되어 있는 멀티라벨 변수입니다.
    예: '현재 시술용, 배아 저장용' → '현재 시술용'=1, '배아 저장용'=1, 나머지=0
    
    MultiLabelBinarizer에 classes를 KNOWN_REASONS로 하드코딩하여, 학습 데이터와
    평가 데이터가 항상 같은 5개 컬럼을 생성하도록 보장합니다.
    이 처리가 데이터 누수 가이드 유형 2를 방지하는 핵심입니다.
    """
    df = df.copy()
    if MULTILABEL_COL not in df.columns:
        return df
    
    # 콤마로 분리하여 리스트로 만든 후, 양 끝의 공백을 제거합니다.
    split_series = df[MULTILABEL_COL].fillna('').apply(
        lambda x: [v.strip() for v in x.split(',') if v.strip()]
    )
    
    # classes 인자를 통해 컬럼 순서와 종류를 학습/평가 데이터에 동일하게 고정합니다.
    mlb = MultiLabelBinarizer(classes=KNOWN_REASONS)
    encoded = pd.DataFrame(
        mlb.fit_transform(split_series),
        columns=[f'목적_{c}' for c in KNOWN_REASONS],
        index=df.index
    )
    return pd.concat([df.drop(columns=[MULTILABEL_COL]), encoded], axis=1)


def preprocess_missing(df, target=None):
    """
    결측치를 고정 상수로 보정합니다.
    
    수치형 결측 → 0
    범주형 결측 → '알 수 없음'
    
    중요: 평균이나 중앙값 등 데이터 통계로 보정하지 않습니다.
    통계 기반 보정은 K-Fold 검증에서 fold 간 정보 유출의 빌미가 될 수 있기 때문입니다.
    또한 17번에 걸친 실험에서 결측치를 별도 플래그로 표현하는 방식(isna 플래그 추가)은
    오히려 모든 시도에서 점수가 떨어지는 결과를 보였습니다.
    """
    df = df.copy()
    
    # 타겟 컬럼이 포함되어 있다면 잠시 분리해 두었다가 마지막에 다시 붙입니다.
    if target and target in df.columns:
        y = df.pop(target)
    else:
        y = None
    
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    num_cols = df.select_dtypes(exclude='object').columns.tolist()
    df[cat_cols] = df[cat_cols].fillna('알 수 없음')
    df[num_cols] = df[num_cols].fillna(0)
    
    if y is not None:
        df[target] = y.values
    return df


def build_features(df, df_raw_ref=None, target=None):
    """
    위 세 단계의 전처리 함수를 차례대로 호출하는 통합 함수입니다.
    학습 데이터와 평가 데이터에 동일하게 적용됩니다.
    
    1) engineer_features    : 도메인 기반 파생 변수 생성
    2) encode_multilabel    : 멀티라벨을 0/1 컬럼으로 분해
    3) preprocess_missing   : 결측 보정
    4) 범주형 컬럼을 category dtype으로 변환 (LightGBM, CatBoost가 자체 처리)
    """
    df = engineer_features(df, df_raw_ref=df_raw_ref)
    df = encode_multilabel(df)
    df = preprocess_missing(df, target=target)
    
    # 범주형 컬럼을 category dtype으로 변환하면 LightGBM과 CatBoost가
    # 별도의 인코딩 없이 내부적으로 효율적으로 처리합니다.
    cat_cols = [c for c in df.select_dtypes(include='object').columns if c != target]
    for col in cat_cols:
        df[col] = df[col].astype('category')
    
    if target and target in df.columns:
        return df.drop(columns=[target]), df[target]
    return df, None


print('전처리 함수 정의가 완료되었습니다.')

In [ ]:
# ============================================================
# 전처리 실행 및 카테고리 일관성 보장
# ============================================================
# 위에서 정의한 build_features 함수를 학습 데이터와 평가 데이터에 동일하게 적용합니다.
# 학습 데이터에는 타겟 컬럼이 포함되어 있고, 평가 데이터에는 없으므로 분리하여 처리합니다.

print('피처 생성 단계를 시작합니다.')
print()

# ------------------------------------------------------------
# 1. 학습/평가 데이터 모두에서 COLS_TO_DROP에 해당하는 컬럼을 제거합니다.
# ------------------------------------------------------------
# errors='ignore'를 지정하면 일부 컬럼이 없어도 오류 없이 통과합니다.
df      = df_raw.drop(columns=COLS_TO_DROP, errors='ignore').copy()
df_test = df_test_raw.drop(columns=COLS_TO_DROP, errors='ignore').copy()

# ------------------------------------------------------------
# 2. 동일한 build_features 함수를 학습/평가 데이터에 각각 적용합니다.
# ------------------------------------------------------------
# 학습 데이터: 타겟 분리됨 → X, y로 반환
# 평가 데이터: 타겟 없음   → X_test만 반환 (두 번째 반환값은 사용하지 않음)
X, y      = build_features(df,      df_raw_ref=df_raw,      target=TARGET)
X_test, _ = build_features(df_test, df_raw_ref=df_test_raw, target=None)

# ------------------------------------------------------------
# 3. 학습 데이터에만 존재하는 컬럼 처리
# ------------------------------------------------------------
# 멀티라벨 인코딩 등의 과정에서 학습 데이터에는 있지만 평가 데이터에는 없는 컬럼이
# 생길 수 있습니다. 이 경우 평가 데이터에 0으로 채운 컬럼을 추가하고,
# 학습 데이터의 컬럼 순서에 맞추어 정렬합니다.
for col in set(X.columns) - set(X_test.columns):
    X_test[col] = 0
X_test = X_test[X.columns]

# ------------------------------------------------------------
# 4. 카테고리 dtype 일관성 보장
# ------------------------------------------------------------
# 같은 범주형 컬럼이라도 학습 데이터와 평가 데이터에서 관측된 값의 종류가 다르면
# pandas가 서로 다른 카테고리 집합으로 인식할 수 있습니다.
# LightGBM은 학습 시 본 적 없는 카테고리가 평가 데이터에서 나타나면 예측이 불안정해질 수 있습니다.
# 이를 방지하기 위해 학습 데이터의 카테고리 집합을 평가 데이터에 강제로 동일하게 적용합니다.
# 평가 데이터에만 존재하는 새로운 값은 NaN으로 처리되어 모델이 자체적으로 다룰 수 있습니다.
cat_cols = [c for c in X.columns if X[c].dtype.name == 'category']
for col in cat_cols:
    X_test[col] = X_test[col].astype(str).astype(
        pd.CategoricalDtype(categories=X[col].cat.categories)
    )

# ------------------------------------------------------------
# 5. 클래스 불균형 보정 가중치 계산
# ------------------------------------------------------------
# scale_pos_weight = 음성 클래스 건수 / 양성 클래스 건수
# LightGBM과 CatBoost에 이 값을 전달하면 양성 샘플의 손실이 가중되어
# 모델이 다수 클래스에 치우치는 현상을 완화합니다.
scale_pos_weight = (y == 0).sum() / (y == 1).sum()

print(f'학습 데이터 변수 수: {X.shape[1]}개')
print(f'평가 데이터 변수 수: {X_test.shape[1]}개')
print(f'두 데이터의 변수 구성이 일치하는지: {list(X.columns) == list(X_test.columns)}')
print(f'클래스 불균형 보정 가중치: {scale_pos_weight:.4f}')

In [ ]:
# ============================================================
# 팀원 피처 5개 추가 (X와 X_test 모두에 동일하게 적용)
# ============================================================
# 누수 방지를 위해 다음 네 가지 원칙을 모두 지킵니다.
#   1. 모든 변환이 행 단위로만 일어납니다 (다른 행의 정보를 사용하지 않음)
#   2. 임계값은 모두 도메인 지식 기반 고정 상수입니다 (데이터 통계 미사용)
#   3. 학습 데이터(X)와 평가 데이터(X_test)에 동일한 코드를 적용합니다
#   4. K-Fold 분할 이전에 추가되지만, 분할 후의 안정성에는 영향을 주지 않습니다
#      (모든 변환이 행 단위라 fold 간 정보 유출이 발생하지 않음)


def add_team_features(X_input):
    """
    팀원이 설계한 임상 도메인 피처 5개를 추가합니다.
    
    매개변수
        X_input : 전처리가 완료된 데이터프레임
    
    반환
        새 변수 5개가 추가된 데이터프레임
    """
    df = X_input.copy()
    
    # ------------------------------------------------------------
    # 변수 1: 배반포_5일차_이식
    # ------------------------------------------------------------
    # 배반포(blastocyst) 단계는 수정 후 5~6일차의 배아 발달 단계입니다.
    # 이 단계까지 키워서 이식하는 방식은 학계에서 일관되게 우위가 보고됩니다 (Roque 2013).
    # 모델이 '5일차 이식'이라는 신호를 명시적으로 학습할 수 있도록 이진화합니다.
    #
    # 정의: 배아 이식 경과일이 정확히 5일이고, 실제로 이식이 진행된 경우 1
    df['배반포_5일차_이식'] = (
        (df['배아 이식 경과일'] == 5) & 
        (df['이식된 배아 수'].fillna(0) > 0)
    ).astype(int)
    
    # ------------------------------------------------------------
    # 변수 2: 젊은_고효율_이식
    # ------------------------------------------------------------
    # 젊은 환자 + 고품질 배아 + 실제 이식 = 임상적으로 최적의 시술 조건입니다.
    # 이 세 조건이 동시에 충족되는 경우를 명시적으로 표시합니다.
    #
    # 정의: Age_Median ≤ 37 + 배아_잉여율 > 0.3 + 이식된 배아 수 ≥ 1
    df['젊은_고효율_이식'] = (
        (df['Age_Median'].fillna(47.5) <= 37) & 
        (df['배아_잉여율'].fillna(0) > 0.3) & 
        (df['이식된 배아 수'].fillna(0) >= 1)
    ).astype(int)
    
    # ------------------------------------------------------------
    # 변수 3: 이식경과일_구간
    # ------------------------------------------------------------
    # 배아 이식 경과일은 본질적으로 구간형 정보입니다.
    # 3일차 이식과 5일차 이식은 임상적으로 다른 의미를 가지므로,
    # 수치형 처리만으로는 잡히지 않는 신호를 구간화로 보강합니다.
    #
    # 구간:
    #   no_transfer : 결측 (이식이 진행되지 않음)
    #   early       : 3일 이하 (분할기 단계 이식)
    #   blastocyst  : 정확히 5일 (배반포 이식)
    #   other       : 그 외 (4일 또는 6일 이상)
    def categorize_transfer_day(x):
        if pd.isna(x):
            return 'no_transfer'
        elif x <= 3:
            return 'early'
        elif x == 5:
            return 'blastocyst'
        else:
            return 'other'
    
    df['이식경과일_구간'] = df['배아 이식 경과일'].apply(categorize_transfer_day).astype('category')
    
    # ------------------------------------------------------------
    # 변수 4: 동결_기증_복합
    # ------------------------------------------------------------
    # 만 45-50세 구간에서 임신 성공률이 일부 회복되는 현상은
    # '기증 난자 + 동결 배아' 조합의 사용 증가가 핵심 원인입니다 (Cozzolino 2024).
    # 이 조합을 명시적으로 표현하여 모델이 해당 패턴을 직접 학습하게 합니다.
    #
    # 정의: 동결 배아 사용 = 1 + 기증난자 = 1
    df['동결_기증_복합'] = (
        (df.get('동결 배아 사용 여부', pd.Series(0, index=df.index)).fillna(0) == 1) & 
        (df['기증난자_여부'] == 1)
    ).astype(int)
    
    # ------------------------------------------------------------
    # 변수 5: 고령_반복시술
    # ------------------------------------------------------------
    # 고령 + 반복 시술 = 임상적으로 가장 어려운 환자군입니다.
    # 이 조합은 Age_Median와 총_시술횟수의 곱셈으로도 부분적으로 학습되지만,
    # 명시적 이진 변수로 추가하여 모델이 해당 패턴을 더 강하게 인지하도록 합니다.
    #
    # 정의: Age_Median ≥ 40 + 총_시술횟수 ≥ 2
    df['고령_반복시술'] = (
        (df['Age_Median'].fillna(47.5) >= 40) & 
        (df['총_시술횟수'].fillna(0) >= 2)
    ).astype(int)
    
    return df


# ------------------------------------------------------------
# 학습 데이터(X)와 평가 데이터(X_test) 모두에 동일하게 적용
# ------------------------------------------------------------
# 같은 함수를 두 데이터에 호출하므로 변환 규칙이 자동으로 일치합니다.
print('팀원 피처 5개 추가 시작')
print(f'  추가 전 X 변수 수:      {X.shape[1]}개')
print(f'  추가 전 X_test 변수 수: {X_test.shape[1]}개')
print()

X      = add_team_features(X)
X_test = add_team_features(X_test)

# ------------------------------------------------------------
# 카테고리 dtype 일관성 보장 (이식경과일_구간이 카테고리형이므로 재처리)
# ------------------------------------------------------------
new_cat_cols = ['이식경과일_구간']
for col in new_cat_cols:
    if col in X.columns:
        X_test[col] = X_test[col].astype(str).astype(
            pd.CategoricalDtype(categories=X[col].cat.categories)
        )

print(f'  추가 후 X 변수 수:      {X.shape[1]}개')
print(f'  추가 후 X_test 변수 수: {X_test.shape[1]}개')
print(f'  변수 일관성 점검:        {list(X.columns) == list(X_test.columns)}')
print()

# ------------------------------------------------------------
# 추가된 5개 변수의 분포 확인
# ------------------------------------------------------------
print('추가 변수의 학습 데이터에서의 분포')
new_features = ['배반포_5일차_이식', '젊은_고효율_이식', '이식경과일_구간', '동결_기증_복합', '고령_반복시술']
for feat in new_features:
    if feat == '이식경과일_구간':
        # 카테고리형은 value_counts
        counts = X[feat].value_counts()
        print(f'  {feat}:')
        for cat_val, cnt in counts.items():
            print(f'    {cat_val:15s}: {cnt:>7,}건 ({cnt/len(X)*100:5.1f}%)')
    else:
        # 이진 변수는 1의 비율
        ones = (X[feat] == 1).sum()
        print(f'  {feat:25s}: 1의 비율 = {ones/len(X)*100:5.1f}% ({ones:,}건)')

print()
print('누수 점검 결과')
print('  ✅ 모든 변수가 행 단위 변환 (다른 행 정보 미사용)')
print('  ✅ 모든 임계값이 고정 상수 (5일, 0.3, 37세, 40세, 2회 — 모두 도메인 기반)')
print('  ✅ Train/Test에 동일한 함수 적용')
print('  ✅ Fold 분할 이전 단계이지만, 행 단위라 fold 간 정보 유출 없음')

In [ ]:
# ============================================================
# 학습 함수 정의
# ============================================================
# LightGBM과 CatBoost를 동일한 검증 구조로 학습하기 위한 함수들을 정의합니다.
# 두 함수 모두 다음 네 가지를 반환합니다.
#   models      : K개 fold에서 학습된 모델 리스트 (변수 중요도 분석에 사용)
#   oof_preds   : 학습 데이터에 대한 OOF 예측 (모델이 학습에 보지 않은 상태에서의 예측)
#   test_preds  : 평가 데이터에 대한 예측 (K개 fold 모델의 평균)
#   oof_auc     : 전체 OOF 예측으로 계산한 AUC


def run_lgbm_oof(X, y, X_test, params, n_splits=5, tag='lgbm'):
    """
    LightGBM을 Stratified K-Fold로 학습하고 OOF 예측 및 평가 데이터 예측을 반환합니다.
    """
    # 클래스 불균형 보정 가중치를 파라미터에 추가합니다.
    scale_pos_weight = (y == 0).sum() / (y == 1).sum()
    params = {**params,
              'scale_pos_weight': scale_pos_weight,
              'objective': 'binary',
              'metric': 'auc',
              'random_state': 42,
              'n_jobs': -1,
              'verbose': -1}

    # Stratified K-Fold: 클래스 비율을 유지하며 fold를 분할합니다.
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    # 결과를 담을 빈 그릇을 준비합니다.
    oof_preds  = np.zeros(len(y))            # 학습 데이터에 대한 OOF 예측
    test_preds = np.zeros(len(X_test))       # 평가 데이터에 대한 K-fold 평균 예측
    cat_feats  = [c for c in X.columns if X[c].dtype.name == 'category']
    models, fold_aucs = [], []

    # K-Fold 반복문
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        model = lgb.LGBMClassifier(**params)
        
        # 학습: 학습 fold로 fit, 검증 fold로 early_stopping
        # early_stopping(100): 검증 AUC가 100번 연속 개선되지 않으면 조기 중단
        model.fit(
            X.iloc[tr_idx], y.iloc[tr_idx],
            eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
            callbacks=[lgb.early_stopping(100, verbose=False),
                       lgb.log_evaluation(500)],
            categorical_feature=cat_feats,
        )
        
        # OOF 예측: 검증 fold 행에 대한 예측만 채워 넣음
        oof_preds[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
        
        # 평가 데이터 예측은 K번 누적된 후 평균이 됩니다 (각 fold에서 1/K씩 더함).
        test_preds += model.predict_proba(X_test)[:, 1] / n_splits
        
        auc = roc_auc_score(y.iloc[val_idx], oof_preds[val_idx])
        fold_aucs.append(auc)
        print(f'  [{tag}] Fold {fold+1}/{n_splits}: AUC={auc:.4f}  iter={model.best_iteration_}')
        models.append(model)

    # 전체 OOF 예측으로 계산한 AUC가 모델의 진짜 성능 지표입니다.
    oof_auc = roc_auc_score(y, oof_preds)
    print(f'\n  [{tag}] OOF AUC={oof_auc:.4f}  (folds={n_splits})')
    return models, oof_preds, test_preds, oof_auc


def run_catboost_oof(X, y, X_test, n_splits=5):
    """
    CatBoost를 동일한 K-Fold 구조로 학습합니다.
    CatBoost는 카테고리를 인덱스 형태로 받기 때문에 dtype을 string으로 변환합니다.
    """
    scale_pos_weight = (y == 0).sum() / (y == 1).sum()
    
    # CatBoost는 카테고리 컬럼의 인덱스를 받습니다 (LightGBM은 이름을 받음).
    cat_feats_idx = [i for i, c in enumerate(X.columns)
                     if X[c].dtype.name == 'category']

    # CatBoost는 category dtype을 직접 다루지 못하므로 string으로 변환합니다.
    X_cb      = X.copy()
    X_test_cb = X_test.copy()
    for col in X.columns:
        if X[col].dtype.name == 'category':
            X_cb[col]      = X_cb[col].astype(str)
            X_test_cb[col] = X_test_cb[col].astype(str)

    # CatBoost 하이퍼파라미터: 17번 실험에서 기본값이 가장 안정적이었습니다.
    # CatBoost Optuna 30회 탐색을 시도해 보았지만 기본 파라미터가 더 좋은 성능을 보였습니다.
    params = {
        'iterations': 2000,
        'learning_rate': 0.05,
        'depth': 6,
        'l2_leaf_reg': 3,
        'scale_pos_weight': scale_pos_weight,
        'eval_metric': 'AUC',
        'random_seed': 42,
        'verbose': False,
        'early_stopping_rounds': 100,
    }
    
    # ------------------------------------------------------------
    # GPU 가속 옵션 (캐글 환경 전용)
    # ------------------------------------------------------------
    # 캐글 노트북에서 GPU 가속기를 활성화한 경우 CatBoost를 GPU로 학습합니다.
    # GPU 사용 시 CPU 대비 약 3~5배 빠르게 학습됩니다.
    # 누수 점검: GPU 사용은 학습 속도만 영향을 주며, fold 분할이나 데이터 처리 방식과 무관합니다.
    if 'USE_GPU' in globals() and USE_GPU:
        params['task_type'] = 'GPU'
        params['devices']   = '0'

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof_preds  = np.zeros(len(y))
    test_preds = np.zeros(len(X_test))
    models, fold_aucs = [], []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_cb, y)):
        model = CatBoostClassifier(**params)
        model.fit(
            X_cb.iloc[tr_idx], y.iloc[tr_idx],
            eval_set=(X_cb.iloc[val_idx], y.iloc[val_idx]),
            cat_features=cat_feats_idx,
        )
        oof_preds[val_idx]  = model.predict_proba(X_cb.iloc[val_idx])[:, 1]
        test_preds         += model.predict_proba(X_test_cb)[:, 1] / n_splits
        auc = roc_auc_score(y.iloc[val_idx], oof_preds[val_idx])
        fold_aucs.append(auc)
        print(f'  [catboost] Fold {fold+1}/{n_splits}: AUC={auc:.4f}')
        models.append(model)

    oof_auc = roc_auc_score(y, oof_preds)
    print(f'\n  [catboost] OOF AUC={oof_auc:.4f}  (folds={n_splits})')
    return models, oof_preds, test_preds, oof_auc


def to_rank(arr):
    """
    예측 확률 배열을 [0, 1] 범위의 순위로 변환합니다.
    
    예를 들어 [0.1, 0.3, 0.2, 0.9] → [0.25, 0.75, 0.5, 1.0]이 됩니다.
    AUC는 순위 기반 지표이므로, 이 변환만으로는 AUC가 바뀌지 않습니다.
    하지만 두 모델의 예측을 평균낼 때는 서로 다른 확률 분포를 동일한 척도로 맞춰주므로
    한 모델이 항상 매우 작은 확률을 출력하는 등의 분포 차이로 인한 왜곡을 줄여 줍니다.
    """
    from scipy.stats import rankdata
    return rankdata(arr) / len(arr)


def weighted_ensemble(oof_list, test_list, y, weights=None, use_rank=False, names=None):
    """
    여러 모델의 OOF 예측과 평가 예측을 가중 평균하는 앙상블 함수입니다.
    
    weights를 지정하지 않으면 각 모델의 OOF AUC에 비례하여 가중치를 자동 계산합니다.
    use_rank=True이면 평균 전에 순위 변환을 거칩니다.
    """
    if use_rank:
        # 순위 변환된 값으로 평균
        oof_list_proc  = [to_rank(o) for o in oof_list]
        test_list_proc = [to_rank(t) for t in test_list]
        print('[앙상블] Rank Averaging 모드')
    else:
        # 원본 확률값으로 평균
        oof_list_proc  = oof_list
        test_list_proc = test_list
        print('[앙상블] 확률값 평균 모드')

    # 가중치 자동 계산: 각 모델의 OOF AUC에 비례하여 부여합니다.
    # 순위 변환은 단조 변환이라 AUC가 바뀌지 않으므로, 가중치 계산은 원본 oof로 합니다.
    if weights is None:
        aucs = [roc_auc_score(y, oof) for oof in oof_list]
        total = sum(aucs)
        weights = [a / total for a in aucs]
        if names:
            for n, w, a in zip(names, weights, aucs):
                print(f'  {n}: weight={w:.4f}, OOF AUC={a:.4f}')

    # 가중 평균 계산
    oof_ensemble  = sum(w * o for w, o in zip(weights, oof_list_proc))
    test_ensemble = sum(w * t for w, t in zip(weights, test_list_proc))
    
    ens_auc = roc_auc_score(y, oof_ensemble)
    print(f'[앙상블] 최종 OOF AUC={ens_auc:.4f}')
    return oof_ensemble, test_ensemble


print('학습 및 앙상블 함수 정의가 완료되었습니다.')

In [ ]:
# ============================================================
# LightGBM 하이퍼파라미터 정의
# ============================================================
# Optuna로 50회 탐색하여 얻은 v4 파라미터가 본 노트북의 기본값입니다.
# 본 데이터에서는 num_leaves=32와 같은 적당한 복잡도가 가장 좋은 성능을 보였고,
# Optuna 100회로 탐색했을 때 얻어진 더 단순한 모델(num_leaves=18)은
# 71개 피처 환경에서 동일한 수준의 성능을 보였습니다.

# v4 파라미터: 본 노트북의 기본 (Optuna 50회 탐색, 5-fold 교차 검증)
v4_params = {
    'learning_rate':     0.032275921555211334,   # 학습률
    'num_leaves':        32,                      # 트리 한 개당 최대 잎 노드 수 (모델 복잡도 통제)
    'min_child_samples': 31,                      # 잎 노드 분할에 필요한 최소 샘플 수
    'subsample':         0.9443563832121702,     # 행 샘플링 비율
    'subsample_freq':    1,                       # subsample을 적용할 주기
    'colsample_bytree':  0.4660046628277167,     # 트리당 사용할 변수 비율
    'reg_alpha':         3.546463779196944,      # L1 정규화
    'reg_lambda':        2.1863992476849217,     # L2 정규화
}

# day4_5 파라미터: Optuna 100회 탐색에서 얻어진 더 보수적인 모델
# 76개 피처 환경에서 탐색되었기에 71개 피처 환경에서는 결과가 다를 수 있습니다.
day45_params = {
    'learning_rate':     0.01227,
    'num_leaves':        18,
    'min_child_samples': 31,                              # 미탐색 → v4 값 재사용
    'subsample':         0.9443563832121702,              # 미탐색 → v4 값 재사용
    'subsample_freq':    1,
    'colsample_bytree':  0.4660046628277167,              # 미탐색 → v4 값 재사용
    'reg_alpha':         3.546463779196944,               # 미탐색 → v4 값 재사용
    'reg_lambda':        8.839,
}

# 토글에 따라 사용할 파라미터를 선택합니다.
if USE_DAY45_PARAMS:
    best_params = day45_params
    print('day4_5 파라미터를 사용합니다 (참고용 비교 실험).')
    print('  주의: 76개 피처 환경에서 탐색된 값이라 본 환경의 결과는 다를 수 있습니다.')
else:
    best_params = v4_params
    print('v4 파라미터를 사용합니다 (LB 0.74191 기록 파라미터).')

# n_estimators는 충분히 큰 값을 두고 early_stopping이 멈출 위치를 찾도록 합니다.
lgbm_params = {**best_params, 'n_estimators': 3000}

print()
print('적용 파라미터')
for k, v in best_params.items():
    print(f'  {k:25s}: {v}')

In [ ]:
# ============================================================
# LightGBM 학습 (K-Fold OOF)
# ============================================================
# N_SPLITS=10이면 약 30분에서 50분, N_SPLITS=5이면 약 15분에서 25분 소요됩니다.
# Colab Pro 환경 기준이며 무료 환경에서는 GPU/세션 끊김에 주의가 필요합니다.

print('=' * 55)
print(f'  LightGBM Stratified {N_SPLITS}-Fold OOF 학습')
print('=' * 55)

lgbm_models, lgbm_oof, lgbm_test, lgbm_auc = run_lgbm_oof(
    X, y, X_test, lgbm_params, n_splits=N_SPLITS, tag='lgbm'
)

In [ ]:
# ============================================================
# CatBoost 학습 (K-Fold OOF)
# ============================================================
# CatBoost는 LightGBM과 비슷한 시간이 소요됩니다.
# 두 모델 모두 끝나면 다음 셀에서 앙상블을 진행합니다.

print('=' * 55)
print(f'  CatBoost Stratified {N_SPLITS}-Fold OOF 학습')
print('=' * 55)

cat_models, cat_oof, cat_test, cat_auc = run_catboost_oof(
    X, y, X_test, n_splits=N_SPLITS
)

In [ ]:
# ============================================================
# 앙상블 및 제출 파일 생성
# ============================================================
# 두 모델의 예측을 가중 평균하여 최종 예측을 만듭니다.
# 가중치는 각 모델의 OOF AUC에 비례하여 자동 계산되며,
# 두 모델의 성능이 거의 비슷하므로 결과적으로 50:50에 가까운 비율이 됩니다.

print('=' * 55)
print(f'  LightGBM과 CatBoost의 앙상블 (use_rank={USE_RANK_AVG})')
print('=' * 55)

oof_ens, test_ens = weighted_ensemble(
    oof_list=[lgbm_oof, cat_oof],
    test_list=[lgbm_test, cat_test],
    y=y,
    use_rank=USE_RANK_AVG,
    names=['LightGBM', 'CatBoost']
)

# ------------------------------------------------------------
# 제출 파일 저장
# ------------------------------------------------------------
# 파일명에 실험 설정을 포함시켜 두면, 여러 번 실험할 때 결과를 추적하기 수월합니다.
suffix = f'_{N_SPLITS}fold'
if USE_RANK_AVG:     suffix += '_rank'
if USE_DAY45_PARAMS: suffix += '_day45'

submission = pd.read_csv(DATA_DIR + 'sample_submission.csv')
submission['probability'] = test_ens
out_name = f'submission_v10_team{suffix}.csv'
submission.to_csv(out_name, index=False)

# ------------------------------------------------------------
# 결과 요약 출력
# ------------------------------------------------------------
print()
print('=== 학습 결과 요약 ===')
print(f'  설정:            N_SPLITS={N_SPLITS}, USE_RANK_AVG={USE_RANK_AVG}, USE_DAY45_PARAMS={USE_DAY45_PARAMS}')
print(f'  LightGBM OOF:    {lgbm_auc:.4f}')
print(f'  CatBoost OOF:    {cat_auc:.4f}')
print(f'  앙상블 OOF:      {roc_auc_score(y, oof_ens):.4f}')
print(f'  제출 파일:       {out_name}')
print()
print()
print('  ── v9 (71개) 비교 ──')
print(f'  v9 (25-fold) OOF:  0.7408')
print(f'  v9 LB:             0.74195')
print(f'  v6 (10-fold) LB:   0.74192')
print(f'  현재 OOF 변화:    {roc_auc_score(y, oof_ens) - 0.7404:+.4f}')
print()
print('제출 파일 미리보기')
print(submission.head(3).to_string())

In [ ]:
# ============================================================
# Fold별 AUC 변동 분석 — 1등과의 차이가 운의 영역임을 입증
# ============================================================
# K-Fold 검증에서 각 fold마다 AUC가 어느 정도 변동하는지 살펴봅니다.
# 이 변동 폭이 1등 팀과 우리 팀의 점수 차이보다 크다면,
# 두 팀의 격차는 모델 실력이 아닌 fold 분할 운으로 충분히 설명될 수 있습니다.
#
# 누수 점검: 학습이 완료된 모델의 OOF 예측만 사용합니다. test 데이터는 사용하지 않습니다.

# ------------------------------------------------------------
# 각 fold의 OOF AUC 계산
# ------------------------------------------------------------
# StratifiedKFold는 random_state를 고정해 두었으므로 학습과 동일한 fold 분할을 재현할 수 있습니다.
skf_for_eval = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

fold_auc_lgbm = []
fold_auc_cat  = []
fold_auc_ens  = []

# 학습 시 저장해 둔 oof 예측을 fold별로 잘라 AUC를 다시 계산합니다.
for fold_idx, (tr_idx, val_idx) in enumerate(skf_for_eval.split(X, y)):
    auc_l = roc_auc_score(y.iloc[val_idx], lgbm_oof[val_idx])
    auc_c = roc_auc_score(y.iloc[val_idx], cat_oof[val_idx])
    auc_e = roc_auc_score(y.iloc[val_idx], oof_ens[val_idx])
    fold_auc_lgbm.append(auc_l)
    fold_auc_cat.append(auc_c)
    fold_auc_ens.append(auc_e)

# 통계량 계산
fold_min   = min(fold_auc_ens)
fold_max   = max(fold_auc_ens)
fold_std   = np.std(fold_auc_ens)
fold_range = fold_max - fold_min

# 비교용 기준값
our_oof = roc_auc_score(y, oof_ens)
our_lb  = 0.74192   # v6 LB 기록
top1_lb = 0.74237   # 1등 LB 기록
gap_to_top1 = top1_lb - our_lb

print(f'fold별 앙상블 AUC: {fold_min:.4f} ~ {fold_max:.4f}  (변동 폭 {fold_range:.4f})')
print(f'fold AUC 표준편차: {fold_std:.4f}')
print(f'우리 LB와 1등 LB 차이: {gap_to_top1:.5f}')
print(f'fold 변동 폭 대비 비율: 1/{fold_range/gap_to_top1:.0f}')
print()

# ------------------------------------------------------------
# 시각화
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(13, 7))

# 막대: 각 fold의 앙상블 AUC
x_pos = np.arange(1, N_SPLITS + 1)
bars = ax.bar(x_pos, fold_auc_ens, width=0.6,
              color='#B3D4FF', edgecolor='gray', linewidth=0.8,
              zorder=3, label=f'fold별 앙상블 AUC')

# 막대 위 수치 표시
for bar, val in zip(bars, fold_auc_ens):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.0005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=8.5,
            bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                      edgecolor='#B3D4FF', linewidth=0.8))

# fold 변동 범위를 회색 음영으로 강조
ax.fill_between([0.4, N_SPLITS + 0.6], [fold_min]*2, [fold_max]*2,
                alpha=0.12, color='gray', zorder=1,
                label=f'fold 변동 범위 ({fold_range:.4f})')

# 우리 OOF 평균 — 파란 실선
ax.axhline(our_oof, color='#1F77B4', linestyle='-', linewidth=2.2,
           zorder=4, label=f'우리 OOF 평균 {our_oof:.4f}')

# 우리 LB — 초록 점선
ax.axhline(our_lb, color='#1D9E75', linestyle='--', linewidth=2.2,
           zorder=4, label=f'우리 LB {our_lb:.4f}')

# 1등 LB — 빨간 점선 (강조)
ax.axhline(top1_lb, color='#D62728', linestyle='--', linewidth=2.2,
           zorder=4, label=f'1등 LB {top1_lb:.4f}')

# 1등과 우리 LB 사이 차이를 화살표로 표시
mid_x = N_SPLITS + 0.7
ax.annotate('',
            xy=(mid_x, top1_lb),
            xytext=(mid_x, our_lb),
            arrowprops=dict(arrowstyle='<->', color='#D62728', lw=1.8))
ax.text(mid_x + 0.1, (top1_lb + our_lb) / 2,
        f'1등과 차이\n{gap_to_top1:.5f}',
        ha='left', va='center', fontsize=9, color='#D62728', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF8E7',
                  edgecolor='#D62728', linewidth=1))

ax.set_xticks(x_pos)
ax.set_xticklabels([f'Fold {i}' for i in x_pos], fontsize=9)
ax.set_ylabel('OOF AUC', fontsize=11)
ax.set_xlabel('Fold 번호', fontsize=11)
ax.set_title(f'Fold별 OOF AUC 변동과 1등 점수와의 비교  ({N_SPLITS}-Fold)',
             fontsize=13, fontweight='bold')
ax.legend(loc='lower left', fontsize=9, ncol=2)

# y축 범위를 fold 변동 범위가 잘 보이도록 조정
y_min = min(fold_auc_ens) - 0.003
y_max = max(top1_lb, max(fold_auc_ens)) + 0.003
ax.set_ylim(y_min, y_max)
ax.set_xlim(0.4, N_SPLITS + 2.0)

ax.yaxis.grid(True, linestyle=':', alpha=0.5, zorder=0)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('fold_variation.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print('=' * 60)
print('  핵심 메시지')
print('=' * 60)
print(f'fold별 AUC가 {fold_min:.4f}부터 {fold_max:.4f}까지 변동 폭 {fold_range:.4f}을 보입니다.')
print(f'우리 LB({our_lb})와 1등 LB({top1_lb})의 차이는 {gap_to_top1:.5f}으로,')
print(f'이는 fold 변동 폭의 약 1/{fold_range/gap_to_top1:.0f} 수준입니다.')
print('즉, 두 팀의 점수 격차는 모델의 본질적 차이라기보다는')
print('fold 분할 운으로 충분히 설명될 수 있는 영역에 속합니다.')


In [ ]:
# ============================================================
# ROC Curve — 분류 모델의 표준 성능 시각화
# ============================================================
# AUC 점수가 높다는 것이 실제로 어떤 분류 곡선의 모양을 의미하는지 시각적으로 확인합니다.
# ROC 곡선은 다양한 임계값에서의 True Positive Rate(민감도)와 
# False Positive Rate(1 - 특이도)의 변화를 보여 줍니다.
# 이상적인 모델은 좌상단 (0, 1)에 가까운 곡선을 그립니다.
#
# 누수 점검: 학습 데이터의 OOF 예측만 사용합니다. test 데이터는 사용하지 않습니다.

from sklearn.metrics import roc_curve

# ------------------------------------------------------------
# 각 모델의 ROC 곡선 좌표 계산
# ------------------------------------------------------------
fpr_lgbm, tpr_lgbm, _ = roc_curve(y, lgbm_oof)
fpr_cat,  tpr_cat,  _ = roc_curve(y, cat_oof)
fpr_ens,  tpr_ens,  _ = roc_curve(y, oof_ens)

auc_lgbm = roc_auc_score(y, lgbm_oof)
auc_cat  = roc_auc_score(y, cat_oof)
auc_ens  = roc_auc_score(y, oof_ens)

# ------------------------------------------------------------
# 그림 작성 — 두 패널
# (왼쪽) 세 모델의 ROC 곡선 비교
# (오른쪽) 앙상블만 크게 + 영역 음영
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))

# 왼쪽: 세 모델 비교
ax1 = axes[0]
ax1.plot(fpr_lgbm, tpr_lgbm, color='#1F77B4', linewidth=1.8,
         label=f'LightGBM 단독 (AUC = {auc_lgbm:.4f})')
ax1.plot(fpr_cat, tpr_cat, color='#FF7F0E', linewidth=1.8,
         label=f'CatBoost 단독 (AUC = {auc_cat:.4f})')
ax1.plot(fpr_ens, tpr_ens, color='#1D9E75', linewidth=2.5,
         label=f'앙상블 (AUC = {auc_ens:.4f})  ★')
ax1.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1.2,
         label='무작위 예측 (AUC = 0.5)')
ax1.set_xlabel('False Positive Rate (1 - 특이도)', fontsize=11)
ax1.set_ylabel('True Positive Rate (민감도)', fontsize=11)
ax1.set_title('세 모델의 ROC 곡선 비교', fontsize=12, fontweight='bold')
ax1.legend(loc='lower right', fontsize=10)
ax1.grid(True, linestyle=':', alpha=0.5)
ax1.set_aspect('equal', adjustable='box')

# 오른쪽: 앙상블만 강조
ax2 = axes[1]
ax2.fill_between(fpr_ens, tpr_ens, alpha=0.18, color='#1D9E75', zorder=2)
ax2.plot(fpr_ens, tpr_ens, color='#1D9E75', linewidth=2.8, zorder=3,
         label=f'앙상블 (AUC = {auc_ens:.4f})')
ax2.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1.2,
         label='무작위 예측 (AUC = 0.5)', zorder=1)

# 임상적 활용 임계값 표시 — recall이 0.7, 0.8인 지점
for target_recall, marker_color in [(0.7, '#D62728'), (0.8, '#9467BD')]:
    # tpr이 target_recall에 가장 가까운 지점 찾기
    idx = np.argmin(np.abs(tpr_ens - target_recall))
    fpr_at = fpr_ens[idx]
    tpr_at = tpr_ens[idx]
    ax2.scatter(fpr_at, tpr_at, s=120, color=marker_color, zorder=5,
                edgecolor='white', linewidth=1.5)
    ax2.annotate(
        f'민감도 {tpr_at:.0%}일 때\n오경보율 {fpr_at:.0%}',
        xy=(fpr_at, tpr_at),
        xytext=(fpr_at + 0.15, tpr_at - 0.08),
        fontsize=9, color=marker_color, fontweight='bold',
        arrowprops=dict(arrowstyle='->', color=marker_color, lw=1.2),
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                  edgecolor=marker_color, linewidth=1)
    )

ax2.set_xlabel('False Positive Rate (1 - 특이도)', fontsize=11)
ax2.set_ylabel('True Positive Rate (민감도)', fontsize=11)
ax2.set_title('앙상블 모델 ROC 곡선  (임상 임계값 예시)', fontsize=12, fontweight='bold')
ax2.legend(loc='lower right', fontsize=10)
ax2.grid(True, linestyle=':', alpha=0.5)
ax2.set_aspect('equal', adjustable='box')

plt.suptitle(f'ROC Curve  ({N_SPLITS}-Fold OOF 예측 기준)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print('해석 안내')
print('  - AUC 0.74라는 수치는 무작위(0.5)와 완벽(1.0) 사이의 중간 정도 분류 능력입니다.')
print('  - 임상에서 이 모델을 활용한다면 임계값 선택이 중요합니다.')
print('  - 예를 들어 민감도(놓치지 않을 비율) 80%를 유지하려면 약 50% 정도의 위양성을 감수해야 합니다.')
print('  - 이는 본 데이터의 본질적 노이즈와 임신 성공의 다요인성에 기인합니다.')


In [ ]:
# ============================================================
# K-fold 평균 피처 중요도
# ============================================================
# 각 fold에서 학습된 LightGBM 모델로부터 Gain importance를 추출하여 평균을 냅니다.
# fold 간 변동의 영향을 줄여 보다 안정적인 변수 순위를 얻기 위함입니다.

# 각 fold 모델의 Gain importance를 행렬로 모읍니다 (K행 × 피처 수 열).
gain_matrix = np.array([
    m.booster_.feature_importance(importance_type='gain')
    for m in lgbm_models
])

# K개 fold의 평균을 사용하여 데이터프레임으로 정리합니다.
fi_gain = pd.DataFrame({
    '피처':  X.columns,
    'Gain':  gain_matrix.mean(axis=0),
}).sort_values('Gain', ascending=False).reset_index(drop=True)

# ------------------------------------------------------------
# 상위 20개 시각화
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 8))
top20 = fi_gain.head(20)
ax.barh(range(20), top20['Gain'].values[::-1],
        color='#B3D4FF', edgecolor='gray', linewidth=0.5)
ax.set_yticks(range(20))
ax.set_yticklabels(top20['피처'].values[::-1], fontsize=9)
ax.set_title(f'LightGBM 피처 중요도 (Gain) 상위 20개 — {N_SPLITS}-fold 평균', fontsize=12)
ax.set_xlabel('Gain (평균)')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print('피처 중요도 상위 10개')
print(fi_gain.head(10).to_string(index=False))
print()
print('해석 참고:')
print('  - 상위 변수의 상당수가 도메인 지식으로 만든 파생 변수입니다.')
print('  - 자가난자×나이, Age×배아수, 배아_잉여율은 새로 만든 변수이며')
print('    원본 컬럼을 그대로 사용한 것보다 더 높은 중요도를 보입니다.')
print('  - 이는 임상 패턴을 변수에 반영한 도메인 기반 설계가 효과적이었음을 시사합니다.')

In [ ]:
# ============================================================
# 상위 20개 중요 변수의 상관 행렬 — 다중공선성 점검
# ============================================================
# 피처 중요도가 높은 변수들 사이의 상관관계를 살펴봅니다.
# 두 변수의 상관계수가 매우 높다(|r| > 0.8)는 것은
# 두 변수가 거의 같은 정보를 담고 있다는 뜻입니다.
#
# 트리 기반 모델(LightGBM, CatBoost)은 다중공선성이 있어도 학습 자체에는 큰 영향이 없으나,
# 어느 변수가 독립적인 신호를 가지는지 파악하는 데 도움이 됩니다.
# 17번 실험에서 변수를 추가할 때마다 점수가 떨어진 이유 중 하나도
# 새 변수가 기존 변수와 정보가 겹쳤기 때문으로 해석할 수 있습니다.
#
# 누수 점검: 학습 데이터(X)의 변수 간 상관만 계산합니다. test는 사용하지 않습니다.

# ------------------------------------------------------------
# 상위 20개 변수 선택
# ------------------------------------------------------------
top20_features = fi_gain.head(20)['피처'].tolist()

# category dtype 변수는 코드로 변환해야 상관 계산 가능
X_top20 = X[top20_features].copy()
for col in X_top20.columns:
    if X_top20[col].dtype.name == 'category':
        # category 코드는 0, 1, 2 ... 로 매핑된 정수입니다.
        X_top20[col] = X_top20[col].cat.codes

# 절댓값 상관 행렬 (방향성 무시, 강도만 본다)
corr_matrix = X_top20.corr().abs()

# ------------------------------------------------------------
# 시각화
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(13, 11))

# 대각선보다 위쪽만 마스킹하여 깔끔하게 표시
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True, fmt='.2f',
    cmap='RdYlGn_r',   # 빨강(높은 상관) → 노랑 → 초록(낮은 상관)
    vmin=0, vmax=1,
    linewidths=0.5, linecolor='white',
    annot_kws={'size': 8, 'weight': 'bold'},
    cbar_kws={'label': '상관계수 (절댓값)', 'shrink': 0.8},
    square=True,
    ax=ax,
)

ax.set_title('상위 20개 중요 변수 간 상관 행렬  (절댓값 기준)',
             fontsize=13, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)

plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# ------------------------------------------------------------
# 강한 상관 페어 자동 추출
# ------------------------------------------------------------
# 본인-본인은 1이므로 제외하고, 절댓값이 높은 페어를 출력합니다.
high_corr_pairs = []
for i in range(len(top20_features)):
    for j in range(i + 1, len(top20_features)):
        r = corr_matrix.iloc[i, j]
        if r > 0.7:
            high_corr_pairs.append((top20_features[i], top20_features[j], r))

high_corr_pairs.sort(key=lambda x: x[2], reverse=True)

print()
print('=' * 60)
print('  상관계수 |r| > 0.7 인 변수 페어')
print('=' * 60)
if high_corr_pairs:
    for var1, var2, r in high_corr_pairs[:15]:
        print(f'  |r|={r:.3f}  |  {var1}  ↔  {var2}')
else:
    print('  높은 상관(|r| > 0.7)을 보이는 페어가 없습니다.')

print()
print('해석 안내')
print('  - 상관이 높은 변수들은 비슷한 정보를 담고 있어, 한쪽만 있어도 같은 효과를 냅니다.')
print('  - 이 데이터에서 자주 함께 등장하는 정보:')
print('    • 배아/난자 수 관련 변수들 (총 생성, 이식, 저장이 함께 움직임)')
print('    • 시술 횟수, 임신 횟수 관련 변수들 (이력 변수들의 군집)')
print('  - 트리 모델은 이러한 중복에 강인하나, 독립적 신호를 가진 변수가 더 가치 있습니다.')


In [ ]:
# ============================================================
# SHAP 라이브러리 준비 및 분석 대상 데이터 추출
# ============================================================
# SHAP은 코랩 기본 환경에 포함되지 않을 수 있어 설치 명령을 한 줄 둡니다.
# 이미 설치되어 있다면 빠르게 통과합니다.
!pip install shap --quiet

import shap

# ------------------------------------------------------------
# 분석에 사용할 모델과 데이터 선택
# ------------------------------------------------------------
# K개 fold 모델 중 가장 OOF AUC가 높은 모델 하나를 선택해 SHAP 분석을 수행합니다.
# 모든 모델을 다 분석하면 시간이 오래 걸리므로, 대표 모델 하나로 충분히 인사이트를 얻을 수 있습니다.
#
# 누수 점검: 학습 데이터(X, y)와 학습된 모델(lgbm_models)만 사용합니다. 
# test 데이터는 사용하지 않습니다.

# 각 fold의 검증 AUC를 다시 계산하여 가장 높은 모델 선정
skf_for_shap = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
fold_aucs_for_shap = []
for fold_idx, (_, val_idx) in enumerate(skf_for_shap.split(X, y)):
    auc = roc_auc_score(y.iloc[val_idx], lgbm_oof[val_idx])
    fold_aucs_for_shap.append(auc)

best_fold = int(np.argmax(fold_aucs_for_shap))
shap_model = lgbm_models[best_fold]
print(f'SHAP 분석에 사용할 모델: Fold {best_fold + 1} '
      f'(검증 AUC = {fold_aucs_for_shap[best_fold]:.4f})')

# ------------------------------------------------------------
# 분석용 샘플 데이터 추출
# ------------------------------------------------------------
# 전체 25만 행에 대해 SHAP 값을 계산하면 5분 이상 걸리고 시각화도 무거워집니다.
# 대신 무작위로 추출한 10,000행으로 분석하면 동일한 인사이트를 빠르게 얻을 수 있습니다.
# 통계적으로 1만 건 정도면 변수의 평균적 영향을 안정적으로 추정할 수 있습니다.
SHAP_SAMPLE_SIZE = 10000
np.random.seed(42)
sample_idx = np.random.choice(len(X), size=min(SHAP_SAMPLE_SIZE, len(X)), replace=False)
X_shap = X.iloc[sample_idx].copy()
y_shap = y.iloc[sample_idx].copy()
oof_shap = oof_ens[sample_idx]

print(f'분석 샘플: {len(X_shap):,}행')
print(f'  실패 (0): {(y_shap == 0).sum():,}건')
print(f'  성공 (1): {(y_shap == 1).sum():,}건')

# ------------------------------------------------------------
# TreeExplainer 생성 (LightGBM은 매우 빠름)
# ------------------------------------------------------------
# TreeExplainer는 트리 모델 전용 SHAP 계산기로, 일반 SHAP보다 수십 배 빠릅니다.
# LightGBM 모델 객체를 받아 분석을 시작합니다.
print('\nSHAP TreeExplainer 생성 중...')
explainer = shap.TreeExplainer(shap_model)

print('SHAP 값 계산 중... (약 1~3분 소요)')
# shap_values는 (n_samples, n_features) 형태의 배열입니다.
# 각 원소는 그 환자의 그 변수가 예측에 미친 영향을 나타냅니다.
shap_values = explainer.shap_values(X_shap)

# 이진 분류에서 일부 SHAP 버전은 [class_0, class_1] 두 배열을 반환합니다.
# 우리는 양성 클래스(임신 성공) 기준으로 분석하므로 [1] 인덱스를 사용합니다.
if isinstance(shap_values, list):
    shap_values = shap_values[1]
    expected_value = explainer.expected_value[1]
else:
    expected_value = explainer.expected_value

print(f'SHAP 값 행렬: {shap_values.shape}')
print(f'기준값 E[f(X)]: {expected_value:.4f}')
print()
print('계산이 완료되었습니다. 다음 셀에서 시각화를 수행합니다.')

In [ ]:
# ============================================================
# SHAP 시각화 1 — Summary Plot (Bar)
# ============================================================
# 각 변수의 SHAP 값 절댓값 평균을 막대 그래프로 보여 줍니다.
# 값이 클수록 그 변수가 예측에 평균적으로 큰 영향을 미친다는 의미입니다.
# 앞 단원의 LightGBM Gain importance와 비교하면, 변수 순위가 비슷한지
# 혹은 다른 양상이 나타나는지 확인할 수 있습니다.

fig = plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values, X_shap,
    plot_type='bar',
    max_display=20,
    show=False,
    color='#B3D4FF',
)
ax = plt.gca()
ax.set_title(f'SHAP 변수 중요도 — 평균 |SHAP value| 상위 20개\n'
             f'(Fold {best_fold + 1} 모델, 샘플 {len(X_shap):,}건 기준)',
             fontsize=12, fontweight='bold', pad=15)
ax.set_xlabel('평균 |SHAP value|  (예측에 미치는 평균적 영향의 크기)', fontsize=10)
plt.tight_layout()
plt.savefig('shap_summary_bar.png', dpi=150, bbox_inches='tight')
plt.show()

# Gain importance와 SHAP 중요도의 상위 5개 비교
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance_df = pd.DataFrame({
    '피처': X_shap.columns,
    'Mean_|SHAP|': mean_abs_shap,
}).sort_values('Mean_|SHAP|', ascending=False).reset_index(drop=True)

print()
print('SHAP 기준 상위 10개 변수')
print(shap_importance_df.head(10).to_string(index=False))
print()
print('Gain 기준 상위 10개와 비교')
comparison = pd.DataFrame({
    'SHAP 순위': range(1, 11),
    'SHAP 변수': shap_importance_df.head(10)['피처'].values,
    'Gain 변수': fi_gain.head(10)['피처'].values,
})
print(comparison.to_string(index=False))

In [ ]:
# ============================================================
# SHAP 시각화 2 — Summary Plot (Beeswarm)
# ============================================================
# Beeswarm 플롯은 SHAP 값의 방향성을 함께 보여 주는 강력한 시각화입니다.
# 한 가로줄이 한 변수에 해당하며, 각 점은 한 환자의 SHAP 값입니다.
#
# 읽는 방법
#   - 점이 오른쪽(양수)에 있을수록 그 환자의 임신 성공 예측을 끌어올림
#   - 점이 왼쪽(음수)에 있을수록 임신 성공 예측을 끌어내림
#   - 색은 변수의 원래 값(빨강 = 큰 값, 파랑 = 작은 값)
#
# 예: '나이'에서 빨간 점들이 왼쪽에 모여 있다면 → 나이가 많을수록 성공 예측이 낮아진다는 의미

fig = plt.figure(figsize=(11, 9))
shap.summary_plot(
    shap_values, X_shap,
    max_display=20,
    show=False,
    plot_size=None,
)
ax = plt.gca()
ax.set_title(f'SHAP 변수 영향 분포 (Beeswarm Plot)\n'
             f'점 색: 변수의 원래 값 크기  /  점 위치: 예측에 미친 영향 (왼쪽 = 성공 확률 낮춤, 오른쪽 = 높임)',
             fontsize=11, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('shap_summary_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print('이 그래프에서 확인할 수 있는 것')
print('  1. 변수의 영향 크기 (가로 폭)')
print('  2. 변수의 영향 방향 (왼쪽/오른쪽)')
print('  3. 변수의 원래 값과 영향의 관계 (점 색)')
print()
print('해석 예시')
print('  - 나이 관련 변수에서 빨간 점(고령)이 왼쪽에, 파란 점(저연령)이 오른쪽에 배치되어 있다면,')
print('    나이가 들수록 임신 성공 예측 확률이 떨어지는 일관된 패턴을 보입니다.')
print('  - 이식된 배아 수에서 빨간 점이 오른쪽에 있다면, 이식 수가 많을수록 성공 예측이 올라간다는 의미입니다.')

In [ ]:
# ============================================================
# SHAP 시각화 3 — Waterfall Plot (임신 성공 예측이 가장 높은 한 환자)
# ============================================================
# 한 환자의 예측이 어떻게 만들어졌는지 변수별로 분해하여 보여 주는 시각화입니다.
# 그래프 아래의 E[f(X)]는 모든 환자의 평균 로짓 예측값(기준선)이며,
# 각 변수가 이 기준선을 얼마나 위/아래로 밀어 올렸는지 누적되어
# 마지막에 그 환자의 최종 예측 f(x)에 도달합니다.
#
# 임신 성공 확률이 가장 높았던 한 환자를 선택해 어떤 변수들이 긍정적으로 작용했는지 확인합니다.

# OOF 예측이 가장 높은 환자 (모델이 가장 확신했던 성공 사례)
top_success_idx = int(np.argmax(oof_shap))
top_success_prob = oof_shap[top_success_idx]
top_success_actual = y_shap.iloc[top_success_idx]

print(f'분석 대상 환자: 샘플 인덱스 {top_success_idx}')
print(f'  모델 예측 확률: {top_success_prob:.4f}')
print(f'  실제 결과:      {"성공" if top_success_actual == 1 else "실패"}')
print()

fig = plt.figure(figsize=(11, 8))

# SHAP 0.40+ 의 Explanation 객체로 만들어 waterfall에 전달
explanation = shap.Explanation(
    values=shap_values[top_success_idx],
    base_values=expected_value,
    data=X_shap.iloc[top_success_idx].values,
    feature_names=list(X_shap.columns),
)

shap.plots.waterfall(explanation, max_display=15, show=False)
plt.title(f'임신 성공 예측이 가장 높았던 환자의 SHAP 분해\n'
          f'(모델 예측 확률 {top_success_prob:.3f}, 실제 결과: {"성공" if top_success_actual == 1 else "실패"})',
          fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('shap_waterfall_success.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print('읽는 방법')
print('  - E[f(X)] = {:.3f}: 모든 환자의 평균 예측 (출발점)'.format(expected_value))
print('  - f(x)   = {:.3f}: 이 환자의 최종 예측'.format(
    expected_value + shap_values[top_success_idx].sum()))
print('  - 빨간 막대 (오른쪽): 성공 확률을 끌어올린 변수')
print('  - 파란 막대 (왼쪽):  성공 확률을 끌어내린 변수')
print()
print('해석: 이 환자에서 가장 큰 양의 영향을 준 변수들이 임상적으로 어떤 의미인지 확인할 수 있습니다.')

In [ ]:
# ============================================================
# SHAP 시각화 4 — Waterfall Plot (임신 성공 예측이 가장 낮은 한 환자)
# ============================================================
# 같은 분석을 임신 성공 예측이 가장 낮았던 환자에 대해 수행합니다.
# 어떤 변수들이 부정적으로 작용했는지 확인하면 모델이 어떤 임상적 패턴을
# 위험 신호로 학습했는지 이해할 수 있습니다.

# OOF 예측이 가장 낮은 환자 (모델이 가장 확신했던 실패 사례)
top_fail_idx = int(np.argmin(oof_shap))
top_fail_prob = oof_shap[top_fail_idx]
top_fail_actual = y_shap.iloc[top_fail_idx]

print(f'분석 대상 환자: 샘플 인덱스 {top_fail_idx}')
print(f'  모델 예측 확률: {top_fail_prob:.4f}')
print(f'  실제 결과:      {"성공" if top_fail_actual == 1 else "실패"}')
print()

fig = plt.figure(figsize=(11, 8))

explanation_fail = shap.Explanation(
    values=shap_values[top_fail_idx],
    base_values=expected_value,
    data=X_shap.iloc[top_fail_idx].values,
    feature_names=list(X_shap.columns),
)

shap.plots.waterfall(explanation_fail, max_display=15, show=False)
plt.title(f'임신 성공 예측이 가장 낮았던 환자의 SHAP 분해\n'
          f'(모델 예측 확률 {top_fail_prob:.3f}, 실제 결과: {"성공" if top_fail_actual == 1 else "실패"})',
          fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('shap_waterfall_fail.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print('두 환자 비교 — 성공 vs 실패의 SHAP 분해는 어떻게 다른가')
print(f'  성공 예측 환자 ({top_success_prob:.3f}): 양의 SHAP 합 = {shap_values[top_success_idx].sum():+.3f}')
print(f'  실패 예측 환자 ({top_fail_prob:.3f}): 양의 SHAP 합 = {shap_values[top_fail_idx].sum():+.3f}')
print()
print('관찰: 두 환자의 SHAP 합이 정반대 방향으로 나타납니다.')
print('해석: 같은 모델이 동일한 변수들을 보고도 환자별 상황에 따라')
print('      양/음 두 방향 모두로 영향을 부여한다는 점이 트리 모델의 비선형 학습 능력을 잘 보여 줍니다.')

In [ ]:
# ============================================================
# 최종 결과 요약 (v10 — 팀원 피처 5개 추가 ablation)
# ============================================================
ensemble_auc = roc_auc_score(y, oof_ens)

print('=' * 60)
print('              v10 학습 결과 요약')
print('=' * 60)
print()
print(f'  데이터:           학습 {X.shape[0]:,}행 × {X.shape[1]}개 변수 (v9 대비 +5)')
print(f'  설정:             N_SPLITS={N_SPLITS}, USE_RANK_AVG={USE_RANK_AVG}')
print()
print(f'  LightGBM OOF:     {lgbm_auc:.4f}')
print(f'  CatBoost OOF:     {cat_auc:.4f}')
print(f'  앙상블 OOF:       {ensemble_auc:.4f}')
print()
print('=' * 60)
print('  v9 (71개) vs v10 (76개) 비교')
print('=' * 60)
print(f'  v9 OOF (71개):    0.7408')
print(f'  v10 OOF (76개):   {ensemble_auc:.4f}')
print(f'  변화:             {ensemble_auc - 0.7408:+.4f}')
print()
print(f'  v9 LB:            0.74195 (확정)')
print(f'  v10 LB:           제출 후 확인')
print()
print('=' * 60)
print('  추가된 팀원 피처 5개 — 중요도 확인')
print('=' * 60)
team_features = ['배반포_5일차_이식', '젊은_고효율_이식', '이식경과일_구간', '동결_기증_복합', '고령_반복시술']
for feat in team_features:
    if feat in fi_gain['피처'].values:
        rank = fi_gain[fi_gain['피처'] == feat].index[0] + 1
        gain = fi_gain[fi_gain['피처'] == feat]['Gain'].values[0]
        print(f'  {feat:20s}: 순위 {rank:>3d}/76, Gain {gain:>10,.0f}')
print()
print(f'  중요도 전체 1위: {fi_gain.iloc[0]["피처"]}')
print(f'  중요도 전체 2위: {fi_gain.iloc[1]["피처"]}')
print(f'  중요도 전체 3위: {fi_gain.iloc[2]["피처"]}')
print()
print('=' * 60)
print('  본 노트북의 데이터 누수 방지 점검')
print('=' * 60)
print('  유형 1 (Train+Test concat):     통과 — 두 데이터를 합치지 않음')
print('  유형 2 (Test get_dummies):      통과 — MultiLabelBinarizer 클래스 하드코딩')
print('  유형 3 (Fold 전 fit):           통과 — Scaler/Imputer 미사용, 고정 상수만 사용')
print('  유형 4 (전체 데이터 selection): 통과 — Clustering/Selection 미사용')
print()
print('  팀원 피처 5개도 모두 동일 기준으로 통과 (행 단위 + 고정 상수)')
print('=' * 60)
print()
print('=' * 60)
print('  결과 해석 가이드')
print('=' * 60)
ensemble_change = ensemble_auc - 0.7408
if ensemble_change >= 0.0001:
    print(f'     OOF +{ensemble_change:.4f} 개선 → 팀원 피처가 의미 있는 신호 추가')
    print(f'     LB 제출을 검토하세요. 개선 가능성이 있습니다.')
elif ensemble_change >= 0:
    print(f'     OOF +{ensemble_change:.4f} 미세 개선 → 거의 동일 수준')
    print(f'     LB 제출 시 변화 미미. 발표 자료에 ablation 결과 포함.')
else:
    print(f'     OOF {ensemble_change:.4f} 하락 → 팀원 피처가 정보 중복으로 작용')
    print(f'     v9 (71개)이 최선임을 다시 확인. "단순함이 강함" 메시지 강화.')
print('=' * 60)

In [ ]:
# ============================================================
# 모든 결과 파일을 ZIP으로 묶기
# ============================================================
import zipfile
import os

zip_path = '/kaggle/working/all_outputs.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for fname in sorted(os.listdir('/kaggle/working/')):
        full_path = f'/kaggle/working/{fname}'
        # 디렉토리는 제외, 파일만 (ZIP 자기 자신도 제외)
        if os.path.isfile(full_path) and fname != 'all_outputs.zip':
            if fname.endswith('.png') or fname.endswith('.csv'):
                zipf.write(full_path, fname)
                print(f"  추가: {fname}")

print()
print(f"ZIP 파일 생성 완료: all_outputs.zip")
print(f"우측 Output 패널에서 'all_outputs.zip' 다운로드하세요.")